# VoltRelay Energy — What is really driving failures, churn and margin?

**Gradient Learnings Data Analytics Hackathon 2026** · Team: **Monish Kannaha S A** and **Lokeshkumar K**

**How to run:** in Google Colab choose *Runtime → Run all*. The first code cell downloads the eight
organiser CSVs from the hackathon Google Drive folder (about 2–3 minutes); a full run takes roughly
10–15 minutes on a standard Colab CPU runtime. Nothing needs to be uploaded by hand.

**How this notebook is organised**

| Part | What it answers |
|---|---|
| 1 · Setup and loading | Download, typed loading of all eight tables |
| 2 · Data understanding | Shape, coverage, keys and relationships |
| 3 · Data cleaning | Every documented data-quality issue: detected, measured, handled, re-checked |
| 4 · Core question 1 | Network performance over time — do the metrics tell the same story? |
| 5 · Core question 2 | Service failures and customer experience |
| 6 · Core question 3 | Station and geographic patterns |
| 7 · Core question 4 | Battery and equipment performance |
| 8 · Core question 5 | Pricing and partner economics |
| 9 · Core question 6 | Root causes of new-rider churn (primary vs secondary drivers) |
| 10 · Decision layer | The ₹ cost of a failed swap, and the four budget proposals scored on one scale |
| 11 · Key findings | What leadership should do, in priority order |

**Conventions.** Money is in Indian rupees (₹; L = lakh, Cr = crore). A *service failure* is an attempt
the network failed to serve: no charged battery, abandoned in the queue, or a system error. Rider
cancellations are reported separately because they are not clearly the network's fault. Every number
quoted in the report is printed by a cell in this notebook.

## 1 · Setup and loading

In [ ]:
import os, re, glob, json, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter, PercentFormatter
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

DRIVE_FOLDER = "https://drive.google.com/drive/folders/1qsjGWrgmEvOaf2Is2fh7RgVEkKd1V3df"
DATA_DIR = Path(os.environ.get("VOLTRELAY_DATA", "/content/voltrelay_data"))
FIG_DIR = Path(os.environ.get("VOLTRELAY_FIGS", "figures"))
OUT_DIR = Path(os.environ.get("VOLTRELAY_OUT", "outputs"))
for d in (DATA_DIR, FIG_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

TABLES = ["swap_events", "station_hourly_status", "riders", "batteries",
          "support_tickets", "stations", "city_daily_context", "fleet_partners"]


def find_table(name):
    """Return every file for a table (plain, gzipped, or split into parts), searched recursively."""
    hits = sorted(p for p in DATA_DIR.rglob(f"{name}*") if p.suffix in {".csv", ".gz"})
    return [p for p in hits if re.match(rf"{name}(_part\d+|\-\d+|_\d+)?\.csv(\.gz)?$", p.name)]


if not all(find_table(t) for t in TABLES):
    import gdown
    print("Downloading the organiser dataset from Google Drive ...")
    gdown.download_folder(DRIVE_FOLDER, output=str(DATA_DIR), quiet=False, remaining_ok=True)

missing = [t for t in TABLES if not find_table(t)]
assert not missing, f"Missing tables after download: {missing}"
for t in TABLES:
    files = find_table(t)
    print(f"{t:<24} {len(files)} file(s), {sum(f.stat().st_size for f in files)/1e6:8.1f} MB")

**Chart style.** One quiet style for every figure: thin marks, hairline solid grids, one highlight
colour when the story is about one thing, and a fixed colour per entity (a city or charger generation
keeps its colour in every chart). No chart uses two y-axes.

In [ ]:
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN, VIOLET, RED = (
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948")
DEEMPH = "#c3c2b7"          # de-emphasised marks
CRITICAL, GOOD = "#d03b3b", "#0ca30c"
SEQ = LinearSegmentedColormap.from_list("seq_blue", [
    "#f3f8fe", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
DIV = LinearSegmentedColormap.from_list("div_blue_red", ["#1c5cab", "#86b6ef", "#f0efec", "#f0a3a2", "#c23636"])

CITY_ORDER = ["Bengaluru", "Delhi NCR", "Hyderabad", "Pune", "Mumbai", "Jaipur"]
CITY_COLOR = dict(zip(CITY_ORDER, [BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREEN]))
GEN_ORDER = ["Gen1", "Gen2", "Gen3"]
GEN_COLOR = {"Gen1": ORANGE, "Gen2": BLUE, "Gen3": AQUA}

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "DejaVu Sans", "font.size": 10, "text.color": INK,
    "axes.edgecolor": AXIS, "axes.labelcolor": INK2, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlepad": 22, "axes.spines.top": False, "axes.spines.right": False,
    "axes.spines.left": False, "axes.grid": True, "axes.grid.axis": "y", "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-",
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK2, "ytick.labelcolor": INK2,
    "xtick.major.size": 0, "ytick.major.size": 0, "lines.linewidth": 2, "lines.solid_capstyle": "round",
    "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110, "savefig.dpi": 160,
    "savefig.bbox": "tight",
})


def titled(ax, title, subtitle=None):
    ax.set_title(title)
    if subtitle:
        ax.text(0, 1.02, subtitle, transform=ax.transAxes, color=INK2, fontsize=9, va="bottom")


def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png")
    plt.show()


def inr(x, digits=1):
    """Format rupees the Indian way: crore (Cr) and lakh (L)."""
    if pd.isna(x):
        return "n/a"
    sign = "-" if x < 0 else ""
    x = abs(x)
    if x >= 1e7:
        return f"{sign}₹{x/1e7:,.{digits}f} Cr"
    if x >= 1e5:
        return f"{sign}₹{x/1e5:,.{digits}f} L"
    return f"{sign}₹{x:,.{0 if x >= 100 else 2}f}"


def pct(x, digits=1):
    return "n/a" if pd.isna(x) else f"{100*x:.{digits}f}%"


pct_axis = PercentFormatter(1.0, decimals=0)
RESULTS = {}      # every headline number used in the report is recorded here and exported at the end


def record(key, value, note=""):
    RESULTS[key] = {"value": (float(value) if isinstance(value, (int, float, np.floating, np.integer)) else value),
                    "note": note}
    return value

**Loading.** Types are set explicitly so that IDs stay text, low-cardinality text becomes categorical
(the 3.9M-row swap table fits comfortably in Colab memory) and timestamps are parsed once.

In [ ]:
def read_table(name, dtypes=None, dates=()):
    frames = []
    for f in find_table(name):
        header = pd.read_csv(f, nrows=0).columns
        dt = {c: t for c, t in (dtypes or {}).items() if c in header}
        frames.append(pd.read_csv(f, dtype=dt, low_memory=False))
    df = pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]
    for c in dates:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    return df


cat, f32 = "category", "float32"
swaps = read_table("swap_events", {
    "event_id": "string", "rider_id": cat, "station_id": cat, "event_type": cat, "attempt_seq": f32,
    "queue_wait_sec": f32, "battery_in_id": cat, "battery_out_id": cat, "soc_in_pct": f32,
    "soh_in_pct": f32, "soc_out_pct": f32, "soh_out_pct": f32, "km_since_last_swap": f32,
    "tariff_code": cat, "list_price_inr": f32, "discount_inr": f32, "amount_charged_inr": f32,
    "payment_mode": cat, "energy_to_recharge_kwh": f32, "station_firmware": cat, "sync_mode": cat,
}, dates=["event_ts"])
hourly = read_table("station_hourly_status", {
    "station_id": cat, "charged_2w_avg": f32, "charged_2w_min": f32, "charged_3w_min": f32,
    "packs_charging": f32, "packs_quarantined": f32, "chargers_online": f32, "ambient_temp_c": f32,
    "cabinet_temp_c": f32, "avg_charge_minutes": f32, "outage_minutes": f32, "grid_kwh": f32,
    "telemetry_status": cat,
}, dates=["hour_start"])
riders = read_table("riders", {"rider_id": "string", "partner_id": "string"}, dates=["signup_date"])
batteries = read_table("batteries", {"battery_id": "string"},
                       dates=["manufacture_date", "commission_date", "retired_date"])
tickets = read_table("support_tickets", {"ticket_id": "string", "rider_id": "string", "station_id": "string",
                                         "battery_id": "string"}, dates=["created_ts"])
stations = read_table("stations", {"station_id": "string"},
                      dates=["commissioned_date", "decommissioned_date", "firmware_updated_date",
                             "competitor_within_1_5km_since"])
city_daily = read_table("city_daily_context", dates=["date"])
partners = read_table("fleet_partners", {"partner_id": "string"}, dates=["contract_start_date", "amendment_date"])

RAW_ROWS = {n: len(df) for n, df in [("swap_events", swaps), ("station_hourly_status", hourly),
            ("riders", riders), ("batteries", batteries), ("support_tickets", tickets),
            ("stations", stations), ("city_daily_context", city_daily), ("fleet_partners", partners)]}
print(pd.Series(RAW_ROWS, name="rows").to_frame().T)
print(f"swap_events in memory: {swaps.memory_usage(deep=True).sum()/1e9:.2f} GB")

## 2 · Data understanding

Before cleaning anything: what does each table cover, do the keys join, and what does a swap attempt
look like? The swap table is the fact table; everything else describes the stations, riders, packs,
partners and conditions around each attempt.

In [ ]:
def coverage(df, col):
    s = df[col].dropna()
    return f"{s.min():%Y-%m-%d} → {s.max():%Y-%m-%d}" if len(s) else "n/a"


overview = pd.DataFrame([
    ("swap_events", len(swaps), swaps.shape[1], "event_id", coverage(swaps, "event_ts")),
    ("station_hourly_status", len(hourly), hourly.shape[1], "station_id + hour_start", coverage(hourly, "hour_start")),
    ("riders", len(riders), riders.shape[1], "rider_id", coverage(riders, "signup_date")),
    ("batteries", len(batteries), batteries.shape[1], "battery_id", coverage(batteries, "commission_date")),
    ("support_tickets", len(tickets), tickets.shape[1], "ticket_id", coverage(tickets, "created_ts")),
    ("stations", len(stations), stations.shape[1], "station_id", coverage(stations, "commissioned_date")),
    ("city_daily_context", len(city_daily), city_daily.shape[1], "city + date", coverage(city_daily, "date")),
    ("fleet_partners", len(partners), partners.shape[1], "partner_id", coverage(partners, "contract_start_date")),
], columns=["table", "rows", "columns", "primary key", "date coverage"])
overview

**Do the keys join?** Every foreign key is checked against its parent table. A key that does not
resolve would silently drop rows in a join, so it is better to know now.

In [ ]:
def fk_share(child, col, parent, pcol):
    vals = child[col].dropna().astype(str)
    return vals.isin(set(parent[pcol].astype(str))).mean() if len(vals) else np.nan


fk = pd.DataFrame([
    ("swap_events.rider_id → riders", fk_share(swaps, "rider_id", riders, "rider_id")),
    ("swap_events.station_id → stations", fk_share(swaps, "station_id", stations, "station_id")),
    ("swap_events.battery_in_id → batteries", fk_share(swaps, "battery_in_id", batteries, "battery_id")),
    ("swap_events.battery_out_id → batteries", fk_share(swaps, "battery_out_id", batteries, "battery_id")),
    ("riders.partner_id → fleet_partners", fk_share(riders, "partner_id", partners, "partner_id")),
    ("support_tickets.rider_id → riders", fk_share(tickets, "rider_id", riders, "rider_id")),
    ("support_tickets.station_id → stations", fk_share(tickets, "station_id", stations, "station_id")),
    ("support_tickets.battery_id → batteries", fk_share(tickets, "battery_id", batteries, "battery_id")),
    ("station_hourly_status.station_id → stations", fk_share(hourly, "station_id", stations, "station_id")),
], columns=["relationship", "share of non-blank keys that resolve"])
fk

**What happens at a cabinet.** The outcome mix of all 3.9M attempts, and which fields are filled for
each outcome (the data dictionary says pricing and battery fields depend on the outcome — this checks it).

In [ ]:
outcome = swaps["event_type"].value_counts().rename("attempts").to_frame()
outcome["share"] = outcome["attempts"] / outcome["attempts"].sum()
fill = swaps.groupby("event_type", observed=True)[
    ["battery_in_id", "battery_out_id", "soh_in_pct", "soh_out_pct", "km_since_last_swap",
     "amount_charged_inr", "energy_to_recharge_kwh"]].agg(lambda s: s.notna().mean())
display(outcome)
display(fill.style.format("{:.0%}"))
charged_on_fail = swaps.loc[swaps["event_type"] != "swap_completed", "amount_charged_inr"].fillna(0)
print(f"Revenue recorded on non-completed attempts: ₹{charged_on_fail.sum():,.0f} "
      f"across {int((charged_on_fail > 0).sum()):,} attempts")

In [ ]:
print("Categorical fields in swap_events:")
for c in ["tariff_code", "payment_mode", "sync_mode", "station_firmware", "attempt_seq"]:
    if c in swaps.columns:
        print(f"  {c:<18}", swaps[c].value_counts(dropna=False).head(8).to_dict())
print("\nStations:")
for c in ["city", "charger_generation", "location_type", "host_type", "expansion_wave", "connectivity_tier"]:
    print(f"  {c:<20}", stations[c].value_counts(dropna=False).to_dict())
print("\nBatteries by supplier:", batteries["supplier"].value_counts().to_dict())
print("Riders by vehicle class:", riders["vehicle_class"].value_counts().to_dict(),
      "| by plan:", riders["plan_type"].value_counts().to_dict())
display(partners)

## 3 · Data cleaning

The organiser documents nine data-quality characteristics. Each one is handled the same way:
**detect** it in the data, **measure** how big it is, **decide** what to do and why, **apply** the fix,
and **re-check** that the fix worked. Every decision is logged in `CLEAN_LOG`, which is printed at the
end of this section so the whole cleaning trail can be audited in one table.

In [ ]:
CLEAN_LOG = []


def log(step, rows, decision):
    CLEAN_LOG.append({"issue": step, "rows affected": int(rows), "decision": decision})


st = stations.copy()
st["station_id"] = st["station_id"].astype(str)
st["is_test"] = st["station_id"].str.startswith("STN-TST")
sw = swaps.copy()
sw["station_id"] = sw["station_id"].astype(str).astype("category")

### 3.1 Test stations
Internal test cabinets (`STN-TST…`) are not customer-facing, so they must not count towards network
performance. First, how big are they really?

In [ ]:
test_ids = set(st.loc[st["is_test"], "station_id"])
is_test_ev = sw["station_id"].isin(test_ids)
test_view = (sw[is_test_ev].groupby("station_id", observed=True)
             .agg(attempts=("event_type", "size"), revenue_inr=("amount_charged_inr", "sum"),
                  first_event=("event_ts", "min"), last_event=("event_ts", "max")))
display(test_view)
print("Hour-of-day of test-station events:",
      sw.loc[is_test_ev, "event_ts"].dt.hour.value_counts().sort_index().to_dict())
record("test_station_events", int(is_test_ev.sum()), "events at STN-TST stations (excluded)")
record("test_station_revenue_inr", float(sw.loc[is_test_ev, "amount_charged_inr"].sum()))
log("Test stations (STN-TST*)", is_test_ev.sum(), "Excluded from every network metric; kept aside for the anomaly list")
sw = sw.loc[~is_test_ev].copy()
sw["station_id"] = sw["station_id"].cat.remove_unused_categories()

### 3.2 Offline-sync near-duplicates
A cabinet with a weak connection can upload the same attempt twice. A duplicate is the same rider,
station, returned pack, issued pack and outcome, logged again within a few minutes, where at least one
copy came through `offline_batch` sync. A genuine retry after a failure would differ in outcome or pack,
so it is not caught by this rule.

In [ ]:
sw = sw.sort_values(["rider_id", "event_ts"], kind="mergesort").reset_index(drop=True)
codes = {c: sw[c].cat.codes.to_numpy() for c in ["rider_id", "station_id", "battery_in_id", "battery_out_id", "event_type"]}
ts_sec = sw["event_ts"].to_numpy().astype("datetime64[s]").astype(np.int64)
gap = np.diff(ts_sec)
same = np.ones(len(gap), dtype=bool)
for c in codes.values():
    same &= c[1:] == c[:-1]
offline = sw["sync_mode"].astype(str).eq("offline_batch").to_numpy()
either_offline = offline[1:] | offline[:-1]
DUP_WINDOW_SEC = 180
cand = same & (gap >= 0) & (gap <= 600)
print("Identical-attempt pairs by gap (seconds), split by whether a copy was offline-synced:")
gap_tab = pd.crosstab(pd.cut(gap[cand], [-1, 10, 30, 60, 120, 180, 300, 600]), either_offline[cand],
                      colnames=["offline copy"])
display(gap_tab)
is_dup = np.r_[False, same & either_offline & (gap >= 0) & (gap <= DUP_WINDOW_SEC)]
dup_by_tier = (pd.DataFrame({"tier": sw["station_id"].map(st.set_index("station_id")["connectivity_tier"]).astype(str),
                             "dup": is_dup}).groupby("tier")["dup"].agg(["sum", "mean"]))
display(dup_by_tier.rename(columns={"sum": "duplicates", "mean": "share of attempts"}))
record("duplicates_removed", int(is_dup.sum()))
log("Offline-sync near-duplicates", is_dup.sum(),
    f"Dropped the later copy when identical within {DUP_WINDOW_SEC}s and one copy was offline-synced")
sw = sw.loc[~is_dup].reset_index(drop=True)

### 3.3 Firmware v3.2.0 clock bug
Stations on firmware v3.2.0 logged times about 5 h 30 m early between 10 March and 14 April 2025 —
exactly the IST offset, i.e. local time was written as if it were UTC. If uncorrected, a morning rush
looks like a 3 a.m. rush. The check below compares the hour-of-day profile of those events with the
same stations' profile in the four weeks before, and with all other stations in the same window.

In [ ]:
fw = sw["station_firmware"].astype(str)
in_window = (sw["event_ts"] >= "2025-03-10") & (sw["event_ts"] < "2025-04-15")
affected = fw.str.contains("3.2.0", regex=False) & in_window
aff_stations = set(sw.loc[affected, "station_id"].astype(str))
before = sw["station_id"].astype(str).isin(aff_stations) & (sw["event_ts"] >= "2025-02-10") & (sw["event_ts"] < "2025-03-10")
others = ~sw["station_id"].astype(str).isin(aff_stations) & in_window


def hour_profile(mask, shift=pd.Timedelta(0)):
    h = (sw.loc[mask, "event_ts"] + shift).dt.hour
    return h.value_counts(normalize=True).reindex(range(24), fill_value=0)


prof = pd.DataFrame({
    "v3.2.0 stations, logged time": hour_profile(affected),
    "v3.2.0 stations, corrected (+5h30m)": hour_profile(affected, pd.Timedelta(hours=5, minutes=30)),
    "same stations, 4 weeks before": hour_profile(before),
    "all other stations, same window": hour_profile(others),
})
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(prof.index, prof.iloc[:, 0], color=RED, label=prof.columns[0])
ax.plot(prof.index, prof.iloc[:, 1], color=BLUE, label=prof.columns[1])
ax.plot(prof.index, prof.iloc[:, 3], color=DEEMPH, label=prof.columns[3])
ax.yaxis.set_major_formatter(pct_axis)
ax.set_xticks(range(0, 24, 3))
ax.set_xlabel("Hour of day")
ax.legend(loc="upper left", ncol=3, bbox_to_anchor=(0, -0.18))
titled(ax, "Hour-of-day profile at v3.2.0 stations: as logged vs corrected",
       "Share of attempts by hour of day, 10 Mar – 14 Apr 2025")
save(fig, "03_clock_bug")
corr_raw = prof.iloc[:, 0].corr(prof.iloc[:, 3])
corr_fix = prof.iloc[:, 1].corr(prof.iloc[:, 3])
print(f"{affected.sum():,} events at {len(aff_stations)} stations affected. Correlation with the normal "
      f"hourly profile: {corr_raw:.2f} as logged → {corr_fix:.2f} after the +5h30m correction.")
record("clock_bug_events", int(affected.sum()))
record("clock_bug_stations", len(aff_stations))
sw.loc[affected, "event_ts"] = sw.loc[affected, "event_ts"] + pd.Timedelta(hours=5, minutes=30)
sw["clock_corrected"] = affected.to_numpy()
log("Firmware v3.2.0 timestamp bug", affected.sum(), "Shifted +5h30m; verified by the hourly-profile match")

### 3.4 City spellings
`riders.home_city` spells the same city several ways. Each variant is mapped to one of the six
network cities with explicit rules, and the full mapping is printed so it can be checked by eye.

In [ ]:
CITY_RULES = [
    ("Bengaluru", ("beng", "bang", "blr", "blore")),
    ("Delhi NCR", ("delhi", "ncr", "gurg", "noida", "dilli", "ndls", "del")),
    ("Hyderabad", ("hyd", "secunderabad", "cyberabad")),
    ("Pune", ("pune", "poona", "pnq")),
    ("Mumbai", ("mum", "bom", "bombay", "thane", "navimumbai")),
    ("Jaipur", ("jaipur", "jaipr", "jpr", "jai")),
]


def canon_city(raw):
    key = re.sub(r"[^a-z]", "", str(raw).lower())
    if not key or key == "nan":
        return np.nan
    for city, keys in CITY_RULES:
        if any(key.startswith(k) or k in key for k in keys):
            return city
    return np.nan


city_map = riders["home_city"].value_counts(dropna=False).rename("riders").to_frame()
city_map["standardised"] = [canon_city(v) for v in city_map.index]
display(city_map)
unmapped = city_map["standardised"].isna() & city_map.index.notna()
if unmapped.any():
    print("Spellings left unmapped (kept as missing):", list(city_map.index[unmapped]))
riders["home_city_clean"] = riders["home_city"].map(canon_city)
variants = riders["home_city"].nunique()
record("city_spelling_variants", variants)
log("Inconsistent city spellings", (riders["home_city"] != riders["home_city_clean"]).sum(),
    f"{variants} spellings mapped to 6 cities by explicit rules")
print("Station and city-context tables use clean names:",
      sorted(st["city"].unique()) == sorted(city_daily["city"].unique()))

### 3.5 Outliers in distance and charge readings
Negative distances and implausibly long ones are odometer resets, not real riding; charge readings a
little above 100% are sensor drift. The plausible ceiling for distance is set from the data itself
(1.5× the 99th percentile for each vehicle class), not guessed.

In [ ]:
vclass = sw["rider_id"].map(riders.set_index("rider_id")["vehicle_class"]).astype(str)
km = sw["km_since_last_swap"]
km_cap = km[km > 0].groupby(vclass[km > 0]).quantile(0.99) * 1.5
bad_km = (km < 0) | (km > vclass.map(km_cap).astype(float))
print("Distance ceiling by vehicle class (km):", km_cap.round(1).to_dict())
print(f"Invalid distances set to missing: {bad_km.sum():,} ({bad_km.mean():.2%} of attempts)")
sw.loc[bad_km, "km_since_last_swap"] = np.nan
over100 = 0
for c in ["soc_in_pct", "soh_in_pct", "soc_out_pct", "soh_out_pct"]:
    n = int((sw[c] > 100).sum())
    over100 += n
    sw[c] = sw[c].clip(upper=100)
neg_wait = int((sw["queue_wait_sec"] < 0).sum())
sw.loc[sw["queue_wait_sec"] < 0, "queue_wait_sec"] = np.nan
log("Distance outliers (odometer resets)", bad_km.sum(), "Negative or > 1.5× p99 set to missing, not dropped")
log("Charge readings above 100%", over100, "Clipped to 100 (calibration drift)")
if neg_wait:
    log("Negative queue waits", neg_wait, "Set to missing")

### 3.6 Missing telemetry
Blank telemetry means *not measured*, not zero. Treating a blank stock reading as zero would invent
stockouts. The table shows where blanks sit: they concentrate at poor-connectivity stations, so every
telemetry metric below is computed over measured hours only.

In [ ]:
hr = hourly.copy()
hr["station_id"] = hr["station_id"].astype(str)
hr = hr[~hr["station_id"].isin(test_ids)].merge(
    st[["station_id", "city", "charger_generation", "location_type", "expansion_wave", "connectivity_tier",
        "slots_2w", "slots_3w", "inventory_target_2w", "inventory_target_3w"]], on="station_id", how="left")
tele = pd.crosstab(hr["connectivity_tier"], hr["telemetry_status"], normalize="index")
display(tele.style.format("{:.1%}"))
blank_stock = hr["charged_2w_min"].isna()
log("Missing telemetry", blank_stock.sum(), "Never zero-filled; metrics use measured hours only")

### 3.7 Satisfaction scores are not missing at random
CSAT is recorded far more often when a ticket is resolved quickly. A plain average therefore
over-represents the happiest tickets. The check below shows the response-rate gradient and compares the
naive average with one re-weighted so that each resolution-time band counts in proportion to its tickets.

In [ ]:
tk = tickets.copy()
tk["res_band"] = pd.cut(tk["resolution_hours"], [-0.01, 4, 12, 24, 48, 96, np.inf],
                        labels=["≤4h", "4–12h", "12–24h", "1–2d", "2–4d", ">4d"])
csat_tab = tk.groupby("res_band", observed=True).agg(
    tickets=("ticket_id", "size"), csat_response_rate=("csat_score", lambda s: s.notna().mean()),
    mean_csat_when_given=("csat_score", "mean"))
display(csat_tab)
w = csat_tab["tickets"] / csat_tab["tickets"].sum()
naive = tk["csat_score"].mean()
reweighted = (csat_tab["mean_csat_when_given"] * w).sum() / w[csat_tab["mean_csat_when_given"].notna()].sum()
print(f"CSAT recorded on {tk['csat_score'].notna().mean():.1%} of tickets. Naive mean {naive:.2f} vs "
      f"re-weighted {reweighted:.2f}; unresolved tickets have no CSAT at all.")
record("csat_naive", naive)
record("csat_reweighted", reweighted)
log("CSAT missing not at random", tk["csat_score"].isna().sum(),
    "Not used as a KPI; quoted only re-weighted by resolution time")

### 3.8 Ticket categories are not always right
Agents file some battery and range complaints under *other* or *app_issue*, and riders often describe
a weak battery as a problem with the vehicle ("gaadi ka pickup kam hai"). A transparent keyword rule
(English and Hinglish) flags comments that are really about range or charge. It is deliberately simple
so it can be audited; sample matches are printed.

In [ ]:
BATTERY_TERMS = [
    r"\bbatt?(e|a)ry\b", r"\brange\b", r"\bcharg", r"\bbackup\b", r"\bdrain", r"\bsoc\b", r"\bpercent|%",
    r"\bkm\b", r"\bkilomet", r"\bkhatam\b", r"\bjaldi\b.*\b(khatam|down|low|ho)", r"\bdischarg",
    r"\bpick ?up\b", r"\bpower\b", r"\bdead\b", r"\bweak\b", r"\bjaldi\b", r"\bkam\b.*\b(chal|range|km)",
    r"\b(ruk|band) (gayi|gaya|ho)", r"\bnahi chal", r"\bchal nahi", r"\bslow\b", r"\bheat|garam|hot\b",
]
battery_re = re.compile("|".join(BATTERY_TERMS), flags=re.IGNORECASE)
tk["mentions_battery"] = tk["rider_comment"].fillna("").str.contains(battery_re)
cat_check = tk.groupby("category").agg(tickets=("ticket_id", "size"),
                                       with_comment=("rider_comment", lambda s: s.notna().mean()),
                                       mentions_battery_or_range=("mentions_battery", "mean"))
display(cat_check.sort_values("tickets", ascending=False))
for c in ["other", "app_issue"]:
    ex = tk.loc[(tk["category"] == c) & tk["mentions_battery"], "rider_comment"].head(4).tolist()
    print(f"Sample '{c}' comments flagged as battery/range: {ex}")
generic = tk["category"].isin(["other", "app_issue"])
tk["category_adj"] = np.where(generic & tk["mentions_battery"], "battery_or_range (re-labelled)", tk["category"])
record("tickets_relabelled", int((generic & tk["mentions_battery"]).sum()))
log("Ticket category mislabelling", (generic & tk["mentions_battery"]).sum(),
    "Generic tickets whose comment is about battery/range re-labelled for analysis (original kept)")

### 3.9 Outcome definitions and further checks
Non-completed attempts are kept (they *are* the failures) but never counted as revenue or delivered
range. The final cell looks for problems the organiser did **not** document.

In [ ]:
et = sw["event_type"].astype(str)
sw["completed"] = et.eq("swap_completed")
sw["stockout"] = et.eq("failed_no_charged_battery")
sw["abandoned"] = et.eq("abandoned_queue")
sw["system_error"] = et.eq("failed_system_error")
sw["cancelled"] = et.eq("cancelled_by_rider")
sw["service_failure"] = sw["stockout"] | sw["abandoned"] | sw["system_error"]
sw.loc[~sw["completed"], ["amount_charged_inr", "energy_to_recharge_kwh"]] = np.nan

bat = batteries.copy()
bat["battery_id"] = bat["battery_id"].astype(str)
retired_on = sw["battery_out_id"].astype(str).map(bat.set_index("battery_id")["retired_date"])
issued_after_retire = sw["completed"] & retired_on.notna() & (sw["event_ts"] > retired_on)
decom = sw["station_id"].astype(str).map(st.set_index("station_id")["decommissioned_date"])
after_decom = decom.notna() & (sw["event_ts"] > decom)
comm = sw["station_id"].astype(str).map(st.set_index("station_id")["commissioned_date"])
before_comm = sw["event_ts"] < comm
signup = sw["rider_id"].astype(str).map(riders.set_index("rider_id")["signup_date"])
before_signup = sw["event_ts"].dt.normalize() < signup
zero_value = sw["completed"] & (sw["amount_charged_inr"].fillna(0) == 0) & (sw["tariff_code"].astype(str) != "PROMO_FREE")
extra = pd.Series({
    "packs issued after their recorded retirement date": int(issued_after_retire.sum()),
    "attempts after the station's decommission date": int(after_decom.sum()),
    "attempts before the station's commission date": int(before_comm.sum()),
    "attempts before the rider's signup date": int(before_signup.sum()),
    "completed paid swaps charged ₹0 (non-promo)": int(zero_value.sum()),
}, name="rows")
display(extra.to_frame())
record("retired_packs_issued", int(issued_after_retire.sum()), "completed swaps issuing a pack after its retired_date")
sw["retired_pack_issued"] = issued_after_retire.to_numpy()
for k, v in extra.items():
    if v:
        log(f"Undocumented: {k}", v, "Flagged and analysed in the anomaly section; rows kept")

**Analysis-ready tables.** Station, rider and calendar attributes are joined once onto the cleaned
swap table, so every later cell works from the same base.

In [ ]:
sw["station_id"] = sw["station_id"].astype(str)
st_cols = ["station_id", "city", "zone", "location_type", "host_type", "expansion_wave", "charger_generation",
           "connectivity_tier", "commissioned_date", "grid_tariff_inr_kwh"]
rd = riders.copy()
rd["rider_id"] = rd["rider_id"].astype(str)
rd_cols = ["rider_id", "partner_id", "vehicle_class", "plan_type", "signup_date", "signup_channel", "home_city_clean"]
sw["rider_id"] = sw["rider_id"].astype(str)
sw = sw.merge(st[st_cols], on="station_id", how="left").merge(rd[rd_cols], on="rider_id", how="left")
for c in ["station_id", "rider_id", "city", "zone", "location_type", "host_type", "expansion_wave",
          "charger_generation", "connectivity_tier", "partner_id", "vehicle_class", "plan_type", "signup_channel"]:
    sw[c] = sw[c].astype("category")
sw["date"] = sw["event_ts"].dt.normalize()
sw["month"] = sw["event_ts"].dt.to_period("M")
sw["hour"] = sw["event_ts"].dt.hour.astype("int8")
sw["dow"] = sw["event_ts"].dt.dayofweek.astype("int8")
sw["is_partner"] = sw["partner_id"].notna()
cd = city_daily.copy()
sw = sw.merge(cd[["city", "date", "max_temp_c", "rainfall_mm", "heat_alert", "is_public_holiday",
                  "competitor_promo_active", "ecommerce_sale_event"]].assign(city=lambda d: d["city"].astype("category")),
              on=["city", "date"], how="left")
hr["date"] = hr["hour_start"].dt.normalize()
hr["month"] = hr["hour_start"].dt.to_period("M")
hr["hour"] = hr["hour_start"].dt.hour

clean_log = pd.DataFrame(CLEAN_LOG)
display(clean_log)
print(f"Cleaned swap table: {len(sw):,} attempts ({RAW_ROWS['swap_events'] - len(sw):,} removed), "
      f"{sw['completed'].sum():,} completed swaps.")
record("attempts_clean", len(sw))
record("completed_clean", int(sw["completed"].sum()))

## 4 · Core question 1 — Network performance over time

*How have completed swaps, revenue, failure rates and contribution margin per swap trended? Do they
tell the same story?*

**Contribution margin per swap** needs a cost model, because the data has costs in three places:

| Cost | How it is computed | Source |
|---|---|---|
| Energy | kWh drawn to recharge the returned pack × that station's grid tariff | swap_events, stations |
| Battery wear | share of the pack's usable life used up per swap × pack purchase cost | batteries, swap_events |
| Site | monthly rent + maintenance of each open station, spread over that station's swaps that month | stations |

*Usable life* runs from the pack's initial state of health (SoH) down to the end-of-life threshold below.
Revenue is the net amount charged, so partner discounts are already netted out.

**Battery wear per swap.** A pack's SoH lost during the data window, divided by the swaps it delivered
in the window, is the health it spends per swap. The end-of-life threshold is taken from the data: the
median SoH of packs retired for health reasons, falling back to the industry-standard 70% if there are
too few.

In [ ]:
issued = sw.loc[sw["completed"] & sw["battery_out_id"].notna(), ["battery_out_id", "event_ts", "soh_out_pct"]]
issued = issued.assign(battery_id=issued["battery_out_id"].astype(str)).sort_values("event_ts")
per_pack = issued.groupby("battery_id").agg(swaps_in_window=("event_ts", "size"),
                                            first_soh_seen=("soh_out_pct", "first"),
                                            first_seen=("event_ts", "min"))
bat = bat.merge(per_pack, left_on="battery_id", right_index=True, how="left")
health_retired = bat["retired_date"].notna() & bat["retirement_reason"].astype(str).str.contains(
    "soh|health|degrad|capacity|end", case=False, regex=True)
EOL_SOH = float(bat.loc[health_retired, "current_soh_pct"].median()) if health_retired.sum() >= 20 else 70.0
EOL_SOH = min(EOL_SOH, 80.0)
start_soh = bat["first_soh_seen"].fillna(bat["initial_soh_pct"])
bat["soh_lost_in_window"] = (start_soh - bat["current_soh_pct"]).clip(lower=0)
bat["soh_per_swap"] = bat["soh_lost_in_window"] / bat["swaps_in_window"]
usable = (bat["initial_soh_pct"] - EOL_SOH).clip(lower=5)
bat["wear_inr_per_swap"] = bat["soh_per_swap"] / usable * bat["purchase_cost_inr"]
print(f"End-of-life SoH threshold used: {EOL_SOH:.1f}% "
      f"({'from ' + str(int(health_retired.sum())) + ' health retirements' if health_retired.sum() >= 20 else 'default'})")
display(bat.groupby("supplier")[["soh_per_swap", "wear_inr_per_swap", "purchase_cost_inr"]].median())
record("eol_soh", EOL_SOH)

In [ ]:
sw["battery_out_str"] = sw["battery_out_id"].astype(str)
sw["wear_inr"] = sw["battery_out_str"].map(bat.set_index("battery_id")["wear_inr_per_swap"]).astype(float)
sw.loc[~sw["completed"], "wear_inr"] = np.nan
sw["energy_inr"] = sw["energy_to_recharge_kwh"].astype(float) * sw["grid_tariff_inr_kwh"].astype(float)

months = pd.period_range(sw["month"].min(), sw["month"].max(), freq="M")
st_open = st[~st["is_test"]].copy()
site_rows = []
for m in months:
    m0, m1 = m.start_time, m.end_time
    open_ = (st_open["commissioned_date"] <= m1) & (st_open["decommissioned_date"].isna() | (st_open["decommissioned_date"] >= m0))
    days_open = ((np.minimum(st_open["decommissioned_date"].fillna(m1), m1) -
                  np.maximum(st_open["commissioned_date"], m0)).dt.days + 1).clip(lower=0)
    frac = days_open / m.days_in_month
    cost = (st_open["monthly_rent_inr"] + st_open["monthly_maintenance_inr"]) * frac
    site_rows.append(pd.DataFrame({"station_id": st_open["station_id"], "month": m, "site_cost_inr": cost.where(open_, 0)}))
site = pd.concat(site_rows, ignore_index=True)
site = site[site["site_cost_inr"] > 0]

done = sw[sw["completed"]]
stm = done.groupby(["station_id", "month"], observed=True).agg(swaps=("completed", "size")).reset_index()
stm["station_id"] = stm["station_id"].astype(str)
site = site.merge(stm, on=["station_id", "month"], how="left").fillna({"swaps": 0})
site_per_swap = site.assign(v=site["site_cost_inr"] / site["swaps"].replace(0, np.nan)).set_index(["station_id", "month"])["v"]
key = pd.MultiIndex.from_arrays([sw["station_id"].astype(str), sw["month"]])
sw["site_inr"] = np.where(sw["completed"], site_per_swap.reindex(key).to_numpy(), np.nan)
sw["contribution_inr"] = sw["amount_charged_inr"] - sw["energy_inr"] - sw["wear_inr"] - sw["site_inr"]
sw["variable_margin_inr"] = sw["amount_charged_inr"] - sw["energy_inr"] - sw["wear_inr"]
idle_site_cost = site.loc[site["swaps"] == 0, "site_cost_inr"].sum()
print(f"Site cost of station-months with zero swaps (not attributable to any swap): {inr(idle_site_cost)}")

**Monthly network scorecard.** One row per month; the four headline metrics are charted below.

In [ ]:
g = sw.groupby("month")
monthly = pd.DataFrame({
    "attempts": g.size(),
    "completed": g["completed"].sum(),
    "revenue_inr": g["amount_charged_inr"].sum(),
    "service_failures": g["service_failure"].sum(),
    "stockouts": g["stockout"].sum(),
    "abandoned": g["abandoned"].sum(),
    "system_errors": g["system_error"].sum(),
    "cancelled": g["cancelled"].sum(),
    "energy_inr": g["energy_inr"].sum(),
    "wear_inr": g["wear_inr"].sum(),
    "site_inr": g["site_inr"].sum(),
    "active_riders": g["rider_id"].nunique(),
    "stations_active": g["station_id"].nunique(),
    "median_wait_min": g["queue_wait_sec"].median() / 60,
})
monthly["service_failure_rate"] = monthly["service_failures"] / monthly["attempts"]
monthly["stockout_rate"] = monthly["stockouts"] / monthly["attempts"]
monthly["abandon_rate"] = monthly["abandoned"] / monthly["attempts"]
for c in ["revenue", "energy", "wear", "site"]:
    monthly[f"{c}_per_swap"] = monthly[f"{c}_inr"] / monthly["completed"]
monthly["contribution_per_swap"] = monthly["revenue_per_swap"] - monthly["energy_per_swap"] - monthly["wear_per_swap"] - monthly["site_per_swap"]
monthly["variable_margin_per_swap"] = monthly["revenue_per_swap"] - monthly["energy_per_swap"] - monthly["wear_per_swap"]
monthly.index = monthly.index.astype(str)
display(monthly[["attempts", "completed", "revenue_inr", "service_failure_rate", "stockout_rate", "abandon_rate",
                 "revenue_per_swap", "energy_per_swap", "wear_per_swap", "site_per_swap", "contribution_per_swap"]])

**When did the business decisions happen?** Rather than hard-coding dates, each decision is located
in the data: the base-price change from the standard list price, the pricing pilot from the first
peak/off-peak tariffs, the new supplier from pack commission dates, the expansion waves from station
commission dates, and the contract amendment from the partner table.

In [ ]:
std = sw[sw["completed"] & sw["tariff_code"].astype(str).eq("STD")]
wk_price = std.groupby([std["event_ts"].dt.to_period("W"), "vehicle_class"], observed=True)["list_price_inr"].median().unstack()
price_change = {}
for vc in wk_price.columns:
    s = wk_price[vc].dropna()
    jumps = s[s.diff().abs() > 0.5]
    price_change[vc] = [(str(p.start_time.date()), float(s.shift(1)[p]), float(v)) for p, v in jumps.items()]
print("Standard list-price changes (week, before, after):", price_change)

tc = sw["tariff_code"].astype(str)
pilot_rows = sw[tc.isin(["PEAK", "OFFPEAK"])]
PILOT_CITIES = sorted(pilot_rows["city"].astype(str).unique())
PILOT_START = pilot_rows["event_ts"].min().normalize()
PILOT_END = pilot_rows["event_ts"].max().normalize()
new_supplier = bat.groupby("supplier")["commission_date"].min().sort_values()
waves = st_open.groupby("expansion_wave")["commissioned_date"].agg(["min", "max", "size"])
amend = partners.loc[partners["amendment_date"].notna(),
                     ["partner_name", "amendment_date", "discount_pct", "discount_pct_after_amendment"]]
print(f"Peak/off-peak pilot: cities {PILOT_CITIES}, {PILOT_START.date()} → {PILOT_END.date()}")
print("First commission date by supplier:", new_supplier.dt.date.to_dict())
display(waves)
display(amend)

BASE_PRICE_DATE = pd.Timestamp(min((c[0][0] for c in price_change.values() if c), default="2024-07-01"))
NEW_SUPPLIER = new_supplier.index[-1]
NEW_SUPPLIER_DATE = new_supplier.iloc[-1]
EVENTS = {"Base price change": BASE_PRICE_DATE, f"{NEW_SUPPLIER} packs arrive": NEW_SUPPLIER_DATE,
          "Pricing pilot": PILOT_START}
for w_name, row in waves.iterrows():
    if not str(w_name).lower().startswith("launch"):
        EVENTS[str(w_name).replace("_", " ")] = row["min"]
for _, r in amend.iterrows():
    EVENTS[f"{r['partner_name']} amendment"] = r["amendment_date"]
record("pilot_cities", ", ".join(PILOT_CITIES))
record("pilot_start", str(PILOT_START.date()))
record("base_price_change", str(BASE_PRICE_DATE.date()))

In [ ]:
def add_events(ax, y_frac=0.97):
    for i, (name, d) in enumerate(sorted(EVENTS.items(), key=lambda kv: kv[1])):
        x = pd.Period(d, "M").to_timestamp()
        ax.axvline(x, color=AXIS, lw=1, zorder=0)
        ax.text(x, y_frac - 0.09 * (i % 3), " " + name, transform=ax.get_xaxis_transform(),
                fontsize=7.5, color=MUTED, va="top")


mts = pd.PeriodIndex(monthly.index, freq="M").to_timestamp()
fig, axes = plt.subplots(2, 2, figsize=(12, 7.2), sharex=True)
panels = [("completed", "Completed swaps per month", lambda v, _: f"{v/1e3:,.0f}K"),
          ("revenue_inr", "Revenue per month", lambda v, _: inr(v, 1)),
          ("service_failure_rate", "Service-failure rate", lambda v, _: f"{100*v:.1f}%"),
          ("contribution_per_swap", "Contribution margin per swap (after energy, wear, site)", lambda v, _: f"₹{v:,.0f}")]
for ax, (col, ttl, fmt) in zip(axes.ravel(), panels):
    ax.plot(mts, monthly[col], color=BLUE)
    ax.scatter(mts[[0, -1]], monthly[col].iloc[[0, -1]], color=BLUE, s=28, zorder=3, edgecolor=SURFACE, linewidth=2)
    for i in (0, -1):
        ax.annotate(fmt(monthly[col].iloc[i], None), (mts[i], monthly[col].iloc[i]), textcoords="offset points",
                    xytext=(0, 8), ha="center", fontsize=8.5, color=INK)
    ax.yaxis.set_major_formatter(FuncFormatter(fmt))
    if col == "contribution_per_swap":
        ax.axhline(0, color=INK2, lw=1)
    add_events(ax)
    ax.set_title(ttl, fontsize=10.5, pad=8)
fig.suptitle("Four headline metrics, month by month", x=0.01, ha="left",
             fontsize=13, fontweight="bold")
fig.tight_layout()
save(fig, "q1_four_metrics")

In [ ]:
q = monthly.copy()
base = q.iloc[:3][["completed", "revenue_inr", "service_failures"]].mean()
idx = q[["completed", "revenue_inr", "service_failures"]] / base * 100
fig, ax = plt.subplots(figsize=(10, 4.2))
for col, color, label in [("service_failures", RED, "Service failures"), ("revenue_inr", BLUE, "Revenue"),
                          ("completed", AQUA, "Completed swaps")]:
    ax.plot(mts, idx[col], color=color, label=label)
    ax.annotate(f"{label}  {idx[col].iloc[-1]/100:.1f}×", (mts[-1], idx[col].iloc[-1]), xytext=(6, 0),
                textcoords="offset points", va="center", fontsize=9, color=INK)
ax.axhline(100, color=AXIS, lw=1)
ax.set_xlim(right=mts[-1] + pd.Timedelta(days=110))
ax.legend(loc="upper left")
titled(ax, "Growth of swaps, revenue and service failures", "Index, Jan–Mar 2024 average = 100")
save(fig, "q1_indexed_growth")
growth = (q.iloc[-3:][["completed", "revenue_inr", "service_failures"]].mean() / base)
record("growth_completed_x", growth["completed"], "last 3 months vs first 3 months")
record("growth_revenue_x", growth["revenue_inr"])
record("growth_failures_x", growth["service_failures"])
record("failure_rate_first3", q.iloc[:3]["service_failures"].sum() / q.iloc[:3]["attempts"].sum())
record("failure_rate_last3", q.iloc[-3:]["service_failures"].sum() / q.iloc[-3:]["attempts"].sum())
record("contribution_per_swap_first3", (q.iloc[:3]["revenue_inr"] - q.iloc[:3][["energy_inr", "wear_inr", "site_inr"]].sum(axis=1)).sum() / q.iloc[:3]["completed"].sum())
record("contribution_per_swap_last3", (q.iloc[-3:]["revenue_inr"] - q.iloc[-3:][["energy_inr", "wear_inr", "site_inr"]].sum(axis=1)).sum() / q.iloc[-3:]["completed"].sum())
print(growth.round(2).to_dict())

**Where did the margin go?** A bridge from the first three months to the last three months, per
completed swap. Each bar is the change in one line of the per-swap P&L.

In [ ]:
def per_swap(block):
    c = block["completed"].sum()
    return pd.Series({k: block[f"{k}_inr"].sum() / c for k in ["revenue", "energy", "wear", "site"]})


p0, p1 = per_swap(q.iloc[:3]), per_swap(q.iloc[-3:])
m0 = p0["revenue"] - p0[["energy", "wear", "site"]].sum()
m1 = p1["revenue"] - p1[["energy", "wear", "site"]].sum()
steps = [("Margin, Jan–Mar 2024", m0, "total"), ("Revenue per swap", p1["revenue"] - p0["revenue"], "delta"),
         ("Energy cost", -(p1["energy"] - p0["energy"]), "delta"), ("Battery wear", -(p1["wear"] - p0["wear"]), "delta"),
         ("Site cost", -(p1["site"] - p0["site"]), "delta"), ("Margin, Apr–Jun 2025", m1, "total")]
fig, ax = plt.subplots(figsize=(9.5, 4.2))
run = 0.0
for i, (lab, v, kind) in enumerate(steps):
    if kind == "total":
        ax.bar(i, v, width=0.55, color=INK2)
        run = v
        y = v
    else:
        ax.bar(i, v, bottom=run, width=0.55, color=GOOD if v >= 0 else CRITICAL)
        run += v
        y = run if v >= 0 else run - v
    ax.annotate(f"{'+' if (kind == 'delta' and v > 0) else ''}₹{v:,.1f}", (i, max(y, run if kind == 'delta' else v)),
                xytext=(0, 4), textcoords="offset points", ha="center", fontsize=9, color=INK)
ax.axhline(0, color=INK2, lw=1)
ax.set_xticks(range(len(steps)), [s[0] for s in steps], fontsize=8.5)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"₹{v:,.0f}"))
titled(ax, "Per-swap margin bridge: what moved between the first and last quarter",
       "₹ per completed swap; green raised margin, red reduced it")
save(fig, "q1_margin_bridge")
bridge = pd.DataFrame(steps, columns=["step", "inr_per_swap", "kind"])
display(bridge)
for lab, v, kind in steps:
    record(f"bridge::{lab}", v)

## 5 · Core question 2 — Service failures and customer experience

*How do queue wait, failed attempts and abandoned swaps relate to station, hour of day, season and
vehicle class? Is service quality spread evenly, or concentrated somewhere specific?*

In [ ]:
fail_mix = sw.groupby(sw["event_ts"].dt.to_period("Q"))[["stockout", "abandoned", "system_error", "cancelled"]].mean()
fail_mix.index = fail_mix.index.astype(str)
display(fail_mix.style.format("{:.2%}"))
by_type = sw[["stockout", "abandoned", "system_error"]].sum()
record("share_failures_stockout", by_type["stockout"] / by_type.sum(), "stockouts as share of service failures")

In [ ]:
heat = sw.groupby(["hour", "month"])["service_failure"].mean().unstack()
fig, ax = plt.subplots(figsize=(12, 5.2))
im = ax.imshow(heat.values, aspect="auto", cmap=SEQ, origin="upper")
ax.set_yticks(range(0, 24, 2), [f"{h:02d}:00" for h in range(0, 24, 2)])
ax.set_xticks(range(len(heat.columns)), [p.strftime("%b\n%y") if p.month in (1, 4, 7, 10) else "" for p in heat.columns])
ax.grid(False)
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01, format=PercentFormatter(1.0, decimals=0))
cb.outline.set_visible(False)
titled(ax, "Service-failure rate by hour of day and month",
       "Service-failure rate by hour of day (rows) and month (columns)")
save(fig, "q2_hour_month_heatmap")
hot = heat.stack().sort_values(ascending=False)
print("Worst hour × month cells:", [(f"{h:02d}:00", str(m), f"{v:.1%}") for (h, m), v in hot.head(5).items()])

**Vehicle class and season.** 3W riders use bigger packs, fewer slots exist for them, and a 3W stockout
strands a cargo vehicle — so the two classes are compared separately. Seasons are grouped as Indian
operating seasons: summer (Apr–Jun), monsoon (Jul–Sep), post-monsoon (Oct–Nov) and winter (Dec–Mar).

In [ ]:
season_map = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Winter", 4: "Summer", 5: "Summer", 6: "Summer",
              7: "Monsoon", 8: "Monsoon", 9: "Monsoon", 10: "Post-monsoon", 11: "Post-monsoon"}
sw["season"] = pd.Categorical(sw["event_ts"].dt.month.map(season_map),
                              ["Winter", "Summer", "Monsoon", "Post-monsoon"], ordered=True)
vc_season = sw.groupby(["vehicle_class", "season"], observed=True).agg(
    attempts=("service_failure", "size"), service_failure_rate=("service_failure", "mean"),
    stockout_rate=("stockout", "mean"), abandon_rate=("abandoned", "mean"),
    median_wait_min=("queue_wait_sec", lambda s: s.median() / 60),
    p90_wait_min=("queue_wait_sec", lambda s: s.quantile(0.9) / 60))
display(vc_season.style.format({"attempts": "{:,.0f}", "service_failure_rate": "{:.2%}", "stockout_rate": "{:.2%}",
                                "abandon_rate": "{:.2%}", "median_wait_min": "{:.1f}", "p90_wait_min": "{:.1f}"}))

fig, ax = plt.subplots(figsize=(9, 3.8))
seasons = ["Winter", "Summer", "Monsoon", "Post-monsoon"]
width = 0.36
for i, (vc, color) in enumerate([("2W", BLUE), ("3W", ORANGE)]):
    vals = [vc_season.loc[(vc, s), "service_failure_rate"] if (vc, s) in vc_season.index else np.nan for s in seasons]
    xs = np.arange(len(seasons)) + (i - 0.5) * (width + 0.03)
    ax.bar(xs, vals, width=width, color=color, label=vc)
    for x, v in zip(xs, vals):
        ax.annotate(f"{v:.1%}", (x, v), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8.5)
ax.set_xticks(range(len(seasons)), seasons)
ax.yaxis.set_major_formatter(pct_axis)
ax.legend(loc="upper right")
titled(ax, "Service-failure rate by season and vehicle class", "Service-failure rate by season")
save(fig, "q2_season_vehicle")

**How concentrated are failures?** If failures were an even, network-wide capacity problem, the
worst stations would carry about the same share of failures as of traffic. The curve compares the two.

In [ ]:
stn = sw.groupby("station_id", observed=True).agg(attempts=("service_failure", "size"),
                                                  failures=("service_failure", "sum"),
                                                  stockouts=("stockout", "sum"))
stn["rate"] = stn["failures"] / stn["attempts"]
stn = stn.sort_values("failures", ascending=False)
cum_fail = stn["failures"].cumsum() / stn["failures"].sum()
cum_att = stn["attempts"].cumsum() / stn["attempts"].sum()
share_st = np.arange(1, len(stn) + 1) / len(stn)
top10 = max(1, int(round(0.10 * len(stn))))
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(np.r_[0, share_st], np.r_[0, cum_fail.values], color=RED, label="Share of service failures")
ax.plot(np.r_[0, share_st], np.r_[0, cum_att.values], color=DEEMPH, label="Share of attempts")
ax.scatter([share_st[top10 - 1]], [cum_fail.iloc[top10 - 1]], color=RED, s=30, zorder=3, edgecolor=SURFACE, linewidth=2)
ax.annotate(f"Worst 10% of stations: {cum_fail.iloc[top10 - 1]:.0%} of failures,\n"
            f"but only {cum_att.iloc[top10 - 1]:.0%} of attempts",
            (share_st[top10 - 1], cum_fail.iloc[top10 - 1]), xytext=(12, -34), textcoords="offset points", fontsize=9)
ax.xaxis.set_major_formatter(pct_axis)
ax.yaxis.set_major_formatter(pct_axis)
ax.set_xlabel("Stations, worst first")
ax.legend(loc="lower right")
ax.grid(axis="x")
titled(ax, "How concentrated are service failures across stations?", "Cumulative share, stations ranked by failures")
save(fig, "q2_concentration")
record("top10pct_stations_failure_share", cum_fail.iloc[top10 - 1])
record("top10pct_stations_attempt_share", cum_att.iloc[top10 - 1])

worst = stn.head(12).join(st.set_index("station_id")[["city", "charger_generation", "location_type", "expansion_wave"]])
display(worst.style.format({"attempts": "{:,.0f}", "failures": "{:,.0f}", "stockouts": "{:,.0f}", "rate": "{:.1%}"}))

**Waiting.** Queue wait is the experience riders feel even when the swap succeeds. The chart compares
the typical (median) wait with the bad-day wait (90th percentile) through the day.

In [ ]:
wait_hr = sw.groupby("hour")["queue_wait_sec"].describe(percentiles=[0.5, 0.9])[["50%", "90%"]] / 60
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(wait_hr.index, wait_hr["90%"], color=BLUE, label="90th percentile")
ax.plot(wait_hr.index, wait_hr["50%"], color=DEEMPH, label="Median")
ax.set_xticks(range(0, 24, 3))
ax.set_xlabel("Hour of day")
ax.set_ylabel("Minutes")
ax.legend(loc="upper left")
titled(ax, "Queue wait through the day", "Queue wait in minutes, all attempts")
save(fig, "q2_wait_by_hour")

## 6 · Core question 3 — Station and geographic patterns

*Stations differ in charger generation, location type and commissioning date, and are unevenly spread
across cities. How do these differences relate to service, battery turnaround and complaints?*

Each attempt is matched to its station's telemetry for that hour (ambient temperature, charged stock,
charge time), so the heat a cabinet was working in is known for every swap.

In [ ]:
tel_cols = ["station_id", "hour_start", "ambient_temp_c", "cabinet_temp_c", "charged_2w_min", "charged_3w_min",
            "packs_quarantined", "chargers_online", "avg_charge_minutes", "outage_minutes"]
sw["hour_start"] = sw["event_ts"].dt.floor("h")
sw = sw.merge(hr[tel_cols].assign(station_id=lambda d: d["station_id"].astype("category")),
              on=["station_id", "hour_start"], how="left")
TEMP_BINS = [-50, 25, 30, 35, 40, 45, 80]
TEMP_LABELS = ["<25°C", "25–30", "30–35", "35–40", "40–45", "≥45°C"]
sw["temp_band"] = pd.cut(sw["ambient_temp_c"], TEMP_BINS, labels=TEMP_LABELS, right=False)
hr["temp_band"] = pd.cut(hr["ambient_temp_c"], TEMP_BINS, labels=TEMP_LABELS, right=False)
print(f"Attempts matched to a measured station-hour temperature: {sw['ambient_temp_c'].notna().mean():.1%}")

In [ ]:
def seg_table(col):
    t = sw.groupby(col, observed=True).agg(stations=("station_id", "nunique"), attempts=("service_failure", "size"),
                                           service_failure_rate=("service_failure", "mean"),
                                           stockout_rate=("stockout", "mean"),
                                           median_wait_min=("queue_wait_sec", lambda s: s.median() / 60))
    return t


tk_st = tk[tk["station_id"].notna()].merge(st[["station_id", "charger_generation", "location_type", "expansion_wave", "city"]],
                                             on="station_id", how="left")
for col in ["charger_generation", "location_type", "expansion_wave", "city"]:
    t = seg_table(col)
    t["tickets_per_1k_attempts"] = tk_st.groupby(col).size().reindex(t.index) / t["attempts"] * 1000
    display(t.style.format({"attempts": "{:,.0f}", "service_failure_rate": "{:.2%}", "stockout_rate": "{:.2%}",
                            "median_wait_min": "{:.1f}", "tickets_per_1k_attempts": "{:.1f}"}))

**Where are the charger generations?** If older cabinets sit in particular cities, a "city problem"
might really be an equipment problem — or the reverse.

In [ ]:
gen_city = pd.crosstab(st_open["city"], st_open["charger_generation"]).reindex(CITY_ORDER).fillna(0).astype(int)
heat_city = city_daily.groupby("city")["max_temp_c"].agg(
    summer_avg_max=lambda s: s[city_daily.loc[s.index, "date"].dt.month.isin([4, 5, 6])].mean(),
    days_above_40=lambda s: (s >= 40).sum())
display(gen_city.join(heat_city))

**Turnaround and stock in the heat.** Hour-level telemetry shows what happens inside the cabinet as it
gets hotter: how long a pack takes to charge, and how often the cabinet has no charged 2W pack at all.

In [ ]:
measured = hr[hr["avg_charge_minutes"].notna() & hr["temp_band"].notna()]
turn = measured.groupby(["temp_band", "charger_generation"], observed=True)["avg_charge_minutes"].median().unstack()
stock_meas = hr[hr["charged_2w_min"].notna() & hr["temp_band"].notna() & (hr["slots_2w"] > 0)]
empty = stock_meas.assign(empty=stock_meas["charged_2w_min"] <= 0).groupby(
    ["temp_band", "charger_generation"], observed=True)["empty"].mean().unstack()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for gen in GEN_ORDER:
    if gen in turn.columns:
        axes[0].plot(range(len(turn)), turn[gen], color=GEN_COLOR[gen], marker="o", ms=5, label=gen,
                     markeredgecolor=SURFACE, markeredgewidth=1.5)
        axes[1].plot(range(len(empty)), empty[gen], color=GEN_COLOR[gen], marker="o", ms=5, label=gen,
                     markeredgecolor=SURFACE, markeredgewidth=1.5)
for ax in axes:
    ax.set_xticks(range(len(turn)), turn.index.astype(str))
    ax.set_xlabel("Ambient temperature")
    ax.legend(loc="upper left")
axes[1].yaxis.set_major_formatter(pct_axis)
titled(axes[0], "Median charge time (minutes)", "By ambient temperature and charger generation")
titled(axes[1], "Hours with zero charged 2W packs", "Share of measured station-hours")
fig.tight_layout()
save(fig, "q3_heat_by_generation")
display(turn.round(0))
display(empty.style.format("{:.1%}"))
for gen in turn.columns:
    record(f"charge_min::{gen}::<25", turn[gen].iloc[0])
    record(f"charge_min::{gen}::hottest", turn[gen].dropna().iloc[-1])
    record(f"empty_hours::{gen}::hottest", empty[gen].dropna().iloc[-1] if gen in empty else np.nan)

**Separating equipment, heat and city.** Gen1 cabinets, hot weather and certain cities overlap, so a
simple table cannot say which one matters. A logistic model of service failure on all of them together
(plus hour of day and vehicle class) holds the others constant. Attempts are aggregated into cells
first, which gives the same estimates as a row-level model but runs in seconds.

In [ ]:
sw["hour_bucket"] = pd.cut(sw["hour"], [-1, 5, 9, 12, 16, 20, 23],
                           labels=["00–05", "06–09", "10–12", "13–16", "17–20", "21–23"])
sw["hot"] = sw["temp_band"].isin(["40–45", "≥45°C"])
cells = (sw.dropna(subset=["temp_band"])
         .groupby(["charger_generation", "hot", "city", "hour_bucket", "vehicle_class", "location_type"], observed=True)
         .agg(n=("service_failure", "size"), k=("service_failure", "sum")).reset_index())
cells = cells[cells["n"] > 0]
cells["rate"] = cells["k"] / cells["n"]
for c in ["charger_generation", "city", "hour_bucket", "vehicle_class", "location_type"]:
    cells[c] = cells[c].astype(str)
cells["hot"] = cells["hot"].astype(int)
fit = smf.glm("rate ~ C(charger_generation, Treatment('Gen2')) * hot + C(city) + C(hour_bucket) + C(vehicle_class)"
              " + C(location_type)", data=cells, family=sm.families.Binomial(), freq_weights=cells["n"]).fit()
orr = pd.DataFrame({"odds_ratio": np.exp(fit.params), "ci_low": np.exp(fit.conf_int()[0]),
                    "ci_high": np.exp(fit.conf_int()[1]), "p_value": fit.pvalues})
display(orr[orr.index.str.contains("generation|hot")].style.format("{:.3f}"))

In [ ]:
def pred_rate(gen, hot):
    d = cells.copy()
    d["charger_generation"], d["hot"] = gen, hot
    return np.average(fit.predict(d), weights=d["n"])


grid = pd.DataFrame({h: {g: pred_rate(g, int(h == "Hot (≥40°C)")) for g in GEN_ORDER}
                     for h in ["Normal (<40°C)", "Hot (≥40°C)"]})
display(grid.style.format("{:.2%}"))
record("adj_fail::Gen1::hot", grid.loc["Gen1", "Hot (≥40°C)"], "model-adjusted service-failure rate")
record("adj_fail::Gen1::normal", grid.loc["Gen1", "Normal (<40°C)"])
record("adj_fail::Gen2::hot", grid.loc["Gen2", "Hot (≥40°C)"])
record("adj_fail::Gen2::normal", grid.loc["Gen2", "Normal (<40°C)"])

**Expansion waves: did new stations go where riders needed them?** For every zone, the service-failure
rate at existing stations in the three months *before* a wave is compared between zones that received
new stations and zones that did not. If the waves targeted pain, the receiving zones should have been
the ones failing.

In [ ]:
wave_rows = []
for wave in [w for w in st_open["expansion_wave"].dropna().unique() if not str(w).lower().startswith("launch")]:
    new = st_open[st_open["expansion_wave"] == wave]
    w0 = new["commissioned_date"].min()
    pre = sw[(sw["event_ts"] >= w0 - pd.DateOffset(months=3)) & (sw["event_ts"] < w0) &
             (sw["commissioned_date"] < w0)]
    z = pre.groupby("zone", observed=True).agg(attempts=("service_failure", "size"), rate=("service_failure", "mean"))
    z["got_new_station"] = z.index.isin(new["zone"])
    post_new = sw[sw["station_id"].astype(str).isin(new["station_id"]) & (sw["event_ts"] < w0 + pd.DateOffset(months=4))]
    days = max(1, (min(sw["event_ts"].max(), w0 + pd.DateOffset(months=4)) - w0).days)
    slots = new["slots_2w"].sum() + new["slots_3w"].sum()
    wave_rows.append({
        "wave": wave, "new stations": len(new), "first opened": w0.date(),
        "pre-wave failure rate, zones that got stations": np.average(z.loc[z["got_new_station"], "rate"], weights=z.loc[z["got_new_station"], "attempts"]) if z["got_new_station"].any() else np.nan,
        "pre-wave failure rate, zones that did not": np.average(z.loc[~z["got_new_station"], "rate"], weights=z.loc[~z["got_new_station"], "attempts"]) if (~z["got_new_station"]).any() else np.nan,
        "new stations in top-quartile failing zones": int(new["zone"].isin(z[z["rate"] >= z["rate"].quantile(0.75)].index).sum()),
        "swaps per slot per day, first 4 months": post_new["completed"].sum() / max(1, slots) / days,
        "host types": new["host_type"].value_counts().to_dict(),
    })
wave_eval = pd.DataFrame(wave_rows).set_index("wave")
display(wave_eval)
network_util = sw["completed"].sum() / (st_open["slots_2w"].sum() + st_open["slots_3w"].sum()) / sw["date"].nunique()
print(f"For comparison, network-wide swaps per slot per day: {network_util:.2f}")

## 7 · Core question 4 — Battery and equipment performance

*How does battery health (state of health, charge cycles, supplier, manufacturing lot) relate to
delivered range and swap frequency? Do any equipment or supplier cohorts stand out?*

In [ ]:
bat["months_in_service"] = ((pd.Timestamp("2025-06-30") - bat["commission_date"]).dt.days / 30.44).clip(lower=0.5)
bat["soh_loss_per_month"] = (bat["initial_soh_pct"] - bat["current_soh_pct"]) / bat["months_in_service"]
lots = bat.groupby(["supplier", "manufacturing_lot"]).agg(
    packs=("battery_id", "size"), commissioned=("commission_date", "min"),
    soh_per_100_swaps=("soh_per_swap", lambda s: 100 * s.median()),
    soh_loss_per_month=("soh_loss_per_month", "median"),
    current_soh=("current_soh_pct", "median"), wear_inr_per_swap=("wear_inr_per_swap", "median")).reset_index()
med = lots["soh_per_100_swaps"].median()
mad = (lots["soh_per_100_swaps"] - med).abs().median() * 1.4826
lots["robust_z"] = (lots["soh_per_100_swaps"] - med) / mad
lots["outlier"] = lots["robust_z"] > 3.5
BAD_LOTS = set(lots.loc[lots["outlier"], "manufacturing_lot"])
display(lots.sort_values("soh_per_100_swaps", ascending=False).head(15).style.format(
    {"soh_per_100_swaps": "{:.3f}", "soh_loss_per_month": "{:.2f}", "current_soh": "{:.1f}",
     "wear_inr_per_swap": "₹{:.2f}", "robust_z": "{:.1f}"}))
print(f"Lots degrading abnormally fast (robust z > 3.5): {sorted(BAD_LOTS)}")
record("bad_lots", ", ".join(sorted(BAD_LOTS)))
record("bad_lot_packs", int(bat["manufacturing_lot"].isin(BAD_LOTS).sum()))
ratio = lots.loc[lots["outlier"], "soh_per_100_swaps"].median() / lots.loc[~lots["outlier"], "soh_per_100_swaps"].median()
record("bad_lot_degradation_multiple", ratio, "median SoH loss per swap, outlier lots vs all other lots")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.4))
ls = lots.sort_values("soh_per_100_swaps").reset_index(drop=True)
colors = [ORANGE if o else DEEMPH for o in ls["outlier"]]
ax.scatter(range(len(ls)), ls["soh_per_100_swaps"], c=colors, s=36, edgecolor=SURFACE, linewidth=1.5, zorder=3)
for i, r in ls[ls["outlier"]].iterrows():
    ax.annotate(f"{r['manufacturing_lot']} ({r['supplier']})", (i, r["soh_per_100_swaps"]), xytext=(-8, 0),
                textcoords="offset points", ha="right", va="center", fontsize=8.5)
ax.set_xticks([])
ax.set_xlabel(f"{len(ls)} manufacturing lots, ranked")
ax.set_ylabel("SoH points lost per 100 swaps")
titled(ax, "Battery degradation rate by manufacturing lot", "Median per lot; highlighted lots are statistical outliers")
save(fig, "q4_lot_degradation")

**Degradation curves.** The state of health of packs *as they are issued to riders*, by months in
service. This uses the swap log itself, so it shows what riders actually received.

In [ ]:
iss = sw.loc[sw["completed"], ["battery_out_str", "event_ts", "soh_out_pct"]].merge(
    bat[["battery_id", "supplier", "manufacturing_lot", "commission_date"]], left_on="battery_out_str",
    right_on="battery_id", how="left")
iss["age_m"] = ((iss["event_ts"] - iss["commission_date"]).dt.days // 30).clip(lower=0)
iss["cohort"] = np.where(iss["manufacturing_lot"].isin(BAD_LOTS), "Outlier lots", iss["supplier"].astype(str))
curve = iss[iss["age_m"] <= 18].groupby(["cohort", "age_m"])["soh_out_pct"].median().unstack(0)
fig, ax = plt.subplots(figsize=(9.5, 4.2))
palette = {"Cellora": BLUE, "Amptek": AQUA, "Kyron": VIOLET, "Outlier lots": ORANGE}
for c in curve.columns:
    ax.plot(curve.index, curve[c], color=palette.get(c, DEEMPH), label=c)
ax.set_xlabel("Months in service")
ax.set_ylabel("Median SoH at issue (%)")
ax.legend(loc="lower left")
titled(ax, "State of health of packs handed to riders, by age", "Median SoH at issue")
save(fig, "q4_degradation_curves")

**Health → range → swap frequency.** A worn pack carries less energy, so the rider covers fewer
kilometres before needing the next swap — and comes back sooner, adding load to the stations.

In [ ]:
ret = sw[sw["battery_in_id"].notna() & sw["km_since_last_swap"].notna() & sw["soh_in_pct"].notna()]
ret = ret.assign(soh_bin=pd.cut(ret["soh_in_pct"], [0, 75, 80, 85, 90, 95, 101],
                                labels=["<75", "75–80", "80–85", "85–90", "90–95", "95–100"]))
rng = ret.groupby(["vehicle_class", "soh_bin"], observed=True)["km_since_last_swap"].agg(["median", "size"]).unstack(0)
display(rng)
sw_sorted = sw[["rider_id", "event_ts", "completed", "soh_out_pct"]].sort_values(["rider_id", "event_ts"])
nxt = sw_sorted.groupby("rider_id", observed=True)["event_ts"].shift(-1)
gap_h = (nxt - sw_sorted["event_ts"]).dt.total_seconds() / 3600
freq = pd.DataFrame({"soh_out": sw_sorted["soh_out_pct"], "gap_h": gap_h})[sw_sorted["completed"].to_numpy()]
freq = freq[(freq["gap_h"] > 0) & (freq["gap_h"] < 72)]
freq["soh_bin"] = pd.cut(freq["soh_out"], [0, 75, 80, 85, 90, 95, 101], labels=["<75", "75–80", "80–85", "85–90", "90–95", "95–100"])
gap_tab = freq.groupby("soh_bin", observed=True)["gap_h"].median()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for vc, color in [("2W", BLUE), ("3W", ORANGE)]:
    if ("median", vc) in rng.columns:
        axes[0].plot(range(len(rng)), rng[("median", vc)], color=color, marker="o", ms=5, label=vc,
                     markeredgecolor=SURFACE, markeredgewidth=1.5)
axes[0].set_xticks(range(len(rng)), rng.index.astype(str))
axes[0].set_xlabel("SoH of returned pack (%)")
axes[0].legend(loc="upper left")
titled(axes[0], "Median km covered on the pack", "By its state of health")
axes[1].plot(range(len(gap_tab)), gap_tab.values, color=BLUE, marker="o", ms=5, markeredgecolor=SURFACE, markeredgewidth=1.5)
axes[1].set_xticks(range(len(gap_tab)), gap_tab.index.astype(str))
axes[1].set_xlabel("SoH of pack received (%)")
titled(axes[1], "Median hours until the rider's next attempt", "By the health of the pack they received")
fig.tight_layout()
save(fig, "q4_range_frequency")
lo, hi = rng.iloc[0], rng.iloc[-1]
record("range_2w_low_soh", rng[("median", "2W")].dropna().iloc[0] if ("median", "2W") in rng else np.nan)
record("range_2w_high_soh", rng[("median", "2W")].dropna().iloc[-1] if ("median", "2W") in rng else np.nan)
record("gap_h_low_soh", gap_tab.dropna().iloc[0])
record("gap_h_high_soh", gap_tab.dropna().iloc[-1])

**Share of swaps served by the outlier lots over time**, and the packs still handed out after their
recorded retirement date (a safety and asset-control question, not only a margin one).

In [ ]:
iss["bad"] = iss["manufacturing_lot"].isin(BAD_LOTS)
bad_share = iss.groupby(iss["event_ts"].dt.to_period("M"))["bad"].mean()
bad_share.index = bad_share.index.astype(str)
print("Share of completed swaps using outlier-lot packs, by month:")
print((bad_share * 100).round(1).to_dict())
record("bad_lot_swap_share_last3", float(bad_share.iloc[-3:].mean()))
ret_iss = sw[sw["retired_pack_issued"]]
print(f"Swaps that issued a pack after its retirement date: {len(ret_iss):,} "
      f"({len(ret_iss)/sw['completed'].sum():.2%} of completed swaps), "
      f"involving {ret_iss['battery_out_str'].nunique():,} packs")
if len(ret_iss):
    display(ret_iss.merge(bat[["battery_id", "supplier", "manufacturing_lot", "retirement_reason"]],
                          left_on="battery_out_str", right_on="battery_id")
            .groupby(["supplier", "retirement_reason"]).size().rename("swaps").to_frame())
print("BMS firmware vs degradation (sanity check):")
display(bat.groupby("bms_firmware")["soh_per_swap"].median().to_frame("median SoH lost per swap"))

## 8 · Core question 5 — Pricing and partner economics

*How do the base price change, the peak/off-peak pilot and individual partner contract terms relate to
swap timing, revenue and contribution margin? Do they affect every rider segment the same way?*

### 8.1 The base price change
Three months before versus three months after the change, by plan type. Partner-billed riders pay a
contracted discount on the list price; pay-as-you-go riders pay list price directly.

In [ ]:
b0, b1 = BASE_PRICE_DATE - pd.DateOffset(months=3), BASE_PRICE_DATE + pd.DateOffset(months=3)
win = sw[(sw["event_ts"] >= b0) & (sw["event_ts"] < b1) & ~sw["city"].astype(str).isin(PILOT_CITIES)].copy()
win["period"] = np.where(win["event_ts"] < BASE_PRICE_DATE, "3 months before", "3 months after")
bp = win.groupby(["plan_type", "period"], observed=True).apply(lambda d: pd.Series({
    "list_price": d.loc[d["completed"], "list_price_inr"].mean(),
    "paid_per_swap": d.loc[d["completed"], "amount_charged_inr"].mean(),
    "swaps_per_rider_week": d["completed"].sum() / d["rider_id"].nunique() / 13,
    "service_failure_rate": d["service_failure"].mean(),
    "margin_per_swap": d.loc[d["completed"], "contribution_inr"].mean(),
})).unstack("period")
display(bp.round(2))

### 8.2 The peak/off-peak pilot — a difference-in-differences test
The pilot changed prices in two cities only, so the other cities show what would have happened anyway.
The estimate is *(change in pilot cities) − (change in other cities)* over the same weeks. Peak hours are
read from the tariff codes themselves (the hours in which PEAK tariffs were charged).

In [ ]:
PEAK_HOURS = sorted(sw.loc[sw["tariff_code"].astype(str) == "PEAK", "hour"].value_counts()
                    .loc[lambda s: s > s.max() * 0.05].index.tolist())
OFFPEAK_HOURS = sorted(sw.loc[sw["tariff_code"].astype(str) == "OFFPEAK", "hour"].value_counts()
                       .loc[lambda s: s > s.max() * 0.05].index.tolist())
print("Peak hours:", PEAK_HOURS, "| Off-peak hours:", OFFPEAK_HOURS)
sw["is_peak_hour"] = sw["hour"].isin(PEAK_HOURS)
sw["pilot_city"] = sw["city"].astype(str).isin(PILOT_CITIES)
pre_start = PILOT_START - pd.DateOffset(weeks=12)
dd = sw[(sw["event_ts"] >= pre_start) & (sw["event_ts"] <= PILOT_END)].copy()
dd["post"] = dd["event_ts"] >= PILOT_START
dd["week"] = dd["event_ts"].dt.to_period("W").astype(str)
cw = dd.groupby(["city", "week"], observed=True).apply(lambda d: pd.Series({
    "peak_share": d["is_peak_hour"].mean(),
    "peak_fail": d.loc[d["is_peak_hour"], "service_failure"].mean(),
    "peak_wait_min": d.loc[d["is_peak_hour"], "queue_wait_sec"].median() / 60,
    "rev_per_swap": d.loc[d["completed"], "amount_charged_inr"].mean(),
    "margin_per_swap": d.loc[d["completed"], "contribution_inr"].mean(),
    "attempts": len(d)})).reset_index()
cw["city"] = cw["city"].astype(str)
cw["treated"] = cw["city"].isin(PILOT_CITIES).astype(int)
cw["post"] = (pd.PeriodIndex(cw["week"], freq="W").start_time >= PILOT_START).astype(int)
did_rows = []
for y in ["peak_share", "peak_fail", "peak_wait_min", "rev_per_swap", "margin_per_swap"]:
    m = smf.wls(f"{y} ~ treated:post + C(city) + C(week)", data=cw.dropna(subset=[y]),
                weights=cw.dropna(subset=[y])["attempts"]).fit(cov_type="HC1")
    pre_mean = cw.loc[(cw["treated"] == 1) & (cw["post"] == 0), y].mean()
    did_rows.append({"outcome": y, "pilot-city baseline": pre_mean, "DiD estimate": m.params["treated:post"],
                     "95% CI low": m.conf_int().loc["treated:post", 0], "95% CI high": m.conf_int().loc["treated:post", 1],
                     "p": m.pvalues["treated:post"]})
did = pd.DataFrame(did_rows).set_index("outcome")
display(did.style.format("{:.4f}"))
for y, r in did.iterrows():
    record(f"did::{y}", r["DiD estimate"], f"baseline {r['pilot-city baseline']:.4f}, p={r['p']:.3f}")

In [ ]:
trend = cw.groupby(["treated", "week"]).apply(lambda d: np.average(d["peak_share"], weights=d["attempts"])).unstack(0)
trend.index = pd.PeriodIndex(trend.index, freq="W").start_time
fig, ax = plt.subplots(figsize=(9.5, 3.8))
ax.plot(trend.index, trend[1], color=BLUE, label="Pilot cities")
ax.plot(trend.index, trend[0], color=DEEMPH, label="Other cities")
ax.axvline(PILOT_START, color=AXIS, lw=1)
ax.text(PILOT_START, 0.98, " Pilot starts", transform=ax.get_xaxis_transform(), fontsize=8, color=MUTED, va="top")
ax.yaxis.set_major_formatter(pct_axis)
ax.legend(loc="lower left")
titled(ax, "Share of attempts made in peak hours", "Weekly, pilot cities vs other cities")
save(fig, "q5_pilot_did")

**Did the pilot reach the riders it was meant to move?** The surcharge only applies to riders who pay
it: independents, and partner riders whose contract makes the surcharge billable. Riders whose employer
absorbs it should not react — a built-in placebo group.

In [ ]:
pmap = partners.set_index("partner_id")["peak_surcharge_billable"].astype(str)
sw["surcharge_exposed"] = np.where(sw["is_partner"], sw["partner_id"].astype(str).map(pmap).eq("Y"), True)
dd["surcharge_exposed"] = np.where(dd["is_partner"], dd["partner_id"].astype(str).map(pmap).eq("Y"), True)
seg = dd.groupby(["surcharge_exposed", "pilot_city", "post"])["is_peak_hour"].mean().unstack("post")
seg["change"] = seg[True] - seg[False]
seg_did = seg["change"].unstack("pilot_city")
seg_did["pilot minus other cities"] = seg_did[True] - seg_did[False]
seg_did.index = seg_did.index.map({True: "Pays the surcharge", False: "Surcharge absorbed by employer"})
display(seg_did.style.format("{:+.2%}"))

base_cong = sw[(sw["event_ts"] < PILOT_START) & sw["is_peak_hour"]].groupby("city", observed=True).agg(
    peak_failure_rate=("service_failure", "mean"), peak_p90_wait_min=("queue_wait_sec", lambda s: s.quantile(0.9) / 60))
base_cong["pilot city"] = base_cong.index.astype(str).isin(PILOT_CITIES)
display(base_cong.sort_values("peak_failure_rate", ascending=False).style.format(
    {"peak_failure_rate": "{:.2%}", "peak_p90_wait_min": "{:.1f}"}))

### 8.3 Partner economics
Revenue is not value. Each partner's swaps carry the same energy, battery-wear and site costs as any
other swap, but earn a discounted price — and some partners pay months later. Contribution per swap is
computed per partner (before and after any contract amendment) with independent riders as a benchmark.
Payment terms are priced at a 12% annual cost of capital (an explicit assumption, shown separately).

In [ ]:
COST_OF_CAPITAL = 0.12
pp = sw[sw["completed"]].copy()
pp["partner_id"] = pp["partner_id"].astype(str).replace({"nan": "Independent", "<NA>": "Independent"})
pinfo = partners.set_index("partner_id")
pp["amended"] = pp["partner_id"].map(pinfo["amendment_date"]).notna() & (
    pp["event_ts"] >= pp["partner_id"].map(pinfo["amendment_date"]))
pp["terms_days"] = pp["partner_id"].map(pinfo["payment_terms_days"]).fillna(0).astype(float)
pp["wc_cost_inr"] = pp["amount_charged_inr"] * COST_OF_CAPITAL * pp["terms_days"] / 365
part = pp.groupby("partner_id").agg(swaps=("completed", "size"), revenue_inr=("amount_charged_inr", "sum"),
                                    list_inr=("list_price_inr", "sum"), discount_inr=("discount_inr", "sum"),
                                    energy_inr=("energy_inr", "sum"), wear_inr=("wear_inr", "sum"),
                                    site_inr=("site_inr", "sum"), wc_inr=("wc_cost_inr", "sum"))
part["contribution_per_swap"] = (part["revenue_inr"] - part[["energy_inr", "wear_inr", "site_inr"]].sum(axis=1)) / part["swaps"]
part["after_payment_terms"] = part["contribution_per_swap"] - part["wc_inr"] / part["swaps"]
part["effective_discount"] = part["discount_inr"] / part["list_inr"]
part = part.join(pinfo[["partner_name", "partner_segment", "vehicle_class", "discount_pct",
                        "discount_pct_after_amendment", "payment_terms_days", "peak_surcharge_billable"]], how="left")
part.loc["Independent", "partner_name"] = "Independent riders"
part = part.sort_values("swaps", ascending=False)
display(part[["partner_name", "partner_segment", "vehicle_class", "swaps", "revenue_inr", "effective_discount",
              "contribution_per_swap", "after_payment_terms", "payment_terms_days"]].style.format(
    {"swaps": "{:,.0f}", "revenue_inr": "₹{:,.0f}", "effective_discount": "{:.1%}",
     "contribution_per_swap": "₹{:.2f}", "after_payment_terms": "₹{:.2f}"}))
largest = part.drop(index="Independent").index[0]
record("largest_partner", str(part.loc[largest, "partner_name"]))
record("largest_partner_swap_share", part.loc[largest, "swaps"] / part["swaps"].sum())
record("largest_partner_contribution_per_swap", part.loc[largest, "contribution_per_swap"])
record("independent_contribution_per_swap", part.loc["Independent", "contribution_per_swap"])

amended_ids = partners.loc[partners["amendment_date"].notna(), "partner_id"].astype(str)
if len(amended_ids):
    ba = pp[pp["partner_id"].isin(amended_ids)].groupby(["partner_id", "amended"]).agg(
        swaps=("completed", "size"), paid_per_swap=("amount_charged_inr", "mean"),
        contribution_per_swap=("contribution_inr", "mean"))
    display(ba)
    for (pid, am), r in ba.iterrows():
        record(f"amend::{pid}::{'after' if am else 'before'}::contribution_per_swap", r["contribution_per_swap"])

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 0.42 * len(part) + 1.2))
pv = part.sort_values("after_payment_terms")
ypos = np.arange(len(pv))
cols = [BLUE if i == largest else (INK2 if i == "Independent" else DEEMPH) for i in pv.index]
ax.barh(ypos, pv["after_payment_terms"], height=0.55, color=cols)
for y, (i, r) in zip(ypos, pv.iterrows()):
    ax.annotate(f"₹{r['after_payment_terms']:.1f}", (r["after_payment_terms"], y), xytext=(4 if r["after_payment_terms"] >= 0 else -4, 0),
                textcoords="offset points", va="center", ha="left" if r["after_payment_terms"] >= 0 else "right", fontsize=8.5)
ax.set_yticks(ypos, [f"{r['partner_name']}  ({r['swaps']/1e3:,.0f}K swaps)" for _, r in pv.iterrows()], fontsize=8.5)
ax.axvline(0, color=INK2, lw=1)
ax.grid(axis="x")
ax.grid(axis="y", visible=False)
titled(ax, "Contribution per swap by customer", "₹ after energy, battery wear, site cost and payment terms; largest partner highlighted")
save(fig, "q5_partner_value")

## 9 · Core question 6 — Why don't new riders come back?

**Definitions.** A *new rider* is anyone who signed up in the data window and made a first attempt on or
before 30 April 2025, so everyone has at least 60 days of follow-up. A rider is **retained** if they made
at least one completed swap in days 31–60 after their first attempt, and **churned** otherwise.

**Early experience** is measured over each rider's first 14 days: failures they hit, how long they
waited, what kind of cabinets and packs they got, whether they had to raise a ticket, what they paid.
Rider attributes (vehicle, plan, partner, channel, city) and competitive context are included too.

**Method.** Three views that must agree before a factor is called *primary*:
1. a logistic regression (effect of each factor holding the others constant, with confidence intervals);
2. a gradient-boosting model with permutation importance (catches non-linear effects; a robustness check);
3. **attributable churn** — how many percentage points of churn the model says would disappear if that
   factor were at its good level for every rider. This combines *how strong* an effect is with *how many
   riders it touches*, which is exactly what separates a primary driver from a secondary one.

In [ ]:
first = sw.groupby("rider_id", observed=True)["event_ts"].min().rename("first_ts")
sw["days_in"] = (sw["event_ts"] - sw["rider_id"].map(first).astype("datetime64[ns]")).dt.days
riders_in = rd.set_index("rider_id")
cohort = first[(first <= "2025-04-30")].to_frame()
cohort = cohort.join(riders_in[["signup_date", "vehicle_class", "plan_type", "partner_id", "signup_channel",
                                "kyc_verified", "age_band", "declared_shift", "home_city_clean"]], how="left")
cohort = cohort[cohort["signup_date"] >= "2024-01-01"]
m2 = sw[(sw["days_in"] >= 30) & (sw["days_in"] < 60) & sw["completed"]].groupby("rider_id", observed=True).size()
cohort["retained"] = cohort.index.isin(m2.index)
cohort["churned"] = ~cohort["retained"]
print(f"New-rider cohort: {len(cohort):,} riders; month-2 retention {cohort['retained'].mean():.1%}")
record("cohort_riders", len(cohort))
record("m2_retention_overall", cohort["retained"].mean())

In [ ]:
early = sw[(sw["days_in"] < 14) & sw["rider_id"].isin(cohort.index)].copy()
early["bad_lot_pack"] = early["battery_out_str"].map(bat.set_index("battery_id")["manufacturing_lot"]).isin(BAD_LOTS)
early["gen1"] = early["charger_generation"].astype(str).eq("Gen1")
eg = early.groupby("rider_id", observed=True)
feat = pd.DataFrame({
    "attempts_14d": eg.size(),
    "service_failures_14d": eg["service_failure"].sum(),
    "any_stockout_14d": eg["stockout"].max().astype(int),
    "any_abandon_14d": eg["abandoned"].max().astype(int),
    "first_attempt_failed": eg["service_failure"].first().astype(int),
    "median_wait_min_14d": eg["queue_wait_sec"].median() / 60,
    "gen1_share_14d": eg["gen1"].mean(),
    "hot_share_14d": eg["hot"].mean(),
    "km_per_pack_14d": eg["km_since_last_swap"].median(),
    "soh_received_14d": eg["soh_out_pct"].mean(),
    "bad_lot_share_14d": eg.apply(lambda d: d.loc[d["completed"], "bad_lot_pack"].mean()),
    "paid_per_swap_14d": eg["amount_charged_inr"].mean(),
    "peak_exposed_share_14d": eg.apply(lambda d: (d["tariff_code"].astype(str) == "PEAK").mean()),
    "competitor_promo_share_14d": eg["competitor_promo_active"].mean(),
    "main_station": eg["station_id"].agg(lambda s: s.value_counts().index[0]),
})
tk14 = tk.merge(first.rename("first_ts").reset_index().assign(rider_id=lambda d: d["rider_id"].astype(str)), on="rider_id")
tk14 = tk14[(tk14["created_ts"] >= tk14["first_ts"]) & (tk14["created_ts"] < tk14["first_ts"] + pd.Timedelta(days=14))]
feat["ticket_14d"] = feat.index.astype(str).isin(set(tk14["rider_id"])).astype(int)
comp_since = st.set_index("station_id")["competitor_within_1_5km_since"]
feat["competitor_nearby"] = (feat["main_station"].astype(str).map(comp_since) <=
                             cohort["first_ts"].reindex(feat.index) + pd.Timedelta(days=14)).astype(int)
feat["main_city"] = feat["main_station"].astype(str).map(st.set_index("station_id")["city"])
cohort = cohort.join(feat, how="inner")
cohort["is_partner"] = cohort["partner_id"].notna().astype(int)
cohort["signup_q"] = cohort["first_ts"].dt.to_period("Q").astype(str)
cohort["service_failure_share_14d"] = cohort["service_failures_14d"] / cohort["attempts_14d"]
display(cohort.groupby("any_stockout_14d")["retained"].agg(["size", "mean"]).rename(
    index={0: "no stockout in first 14 days", 1: "≥1 stockout in first 14 days"}, columns={"size": "riders", "mean": "retention"}))

**Retention by signup month**, split by whether the rider hit a stockout in their first two weeks.

In [ ]:
cohort["signup_m"] = cohort["first_ts"].dt.to_period("M")
rt = cohort.groupby(["signup_m", "any_stockout_14d"])["retained"].mean().unstack()
rt_all = cohort.groupby("signup_m")["retained"].mean()
exp_share = cohort.groupby("signup_m")["any_stockout_14d"].mean()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
x = rt.index.to_timestamp()
axes[0].plot(x, rt[0], color=BLUE, label="No early stockout")
axes[0].plot(x, rt[1], color=ORANGE, label="≥1 early stockout")
axes[0].plot(x, rt_all.values, color=INK2, lw=1.2, label="All new riders")
axes[0].yaxis.set_major_formatter(pct_axis)
axes[0].legend(loc="lower left")
titled(axes[0], "Month-2 retention by signup month", "Share of new riders still swapping in days 31–60")
axes[1].plot(x, exp_share.values, color=ORANGE)
axes[1].yaxis.set_major_formatter(pct_axis)
titled(axes[1], "New riders hitting a stockout in their first 14 days", "Share of each signup month")
fig.tight_layout()
save(fig, "q6_retention_trend")
record("m2_retention_first_q", cohort[cohort["signup_q"] == cohort["signup_q"].min()]["retained"].mean())
record("m2_retention_last_q", cohort[cohort["signup_q"] == cohort["signup_q"].max()]["retained"].mean())
record("retention_no_stockout", cohort.loc[cohort["any_stockout_14d"] == 0, "retained"].mean())
record("retention_with_stockout", cohort.loc[cohort["any_stockout_14d"] == 1, "retained"].mean())

**The model.** Continuous factors are standardised, so each odds ratio is the effect of a one-standard-
deviation change; binary factors compare yes vs no. An odds ratio above 1 means *more* churn.

In [ ]:
CONT = ["attempts_14d", "service_failure_share_14d", "median_wait_min_14d", "gen1_share_14d", "hot_share_14d",
        "km_per_pack_14d", "soh_received_14d", "bad_lot_share_14d", "paid_per_swap_14d", "peak_exposed_share_14d",
        "competitor_promo_share_14d"]
BIN = ["any_stockout_14d", "any_abandon_14d", "first_attempt_failed", "ticket_14d", "competitor_nearby"]
CATS = ["vehicle_class", "plan_type", "signup_channel", "main_city", "signup_q"]
X = cohort[CONT + BIN + CATS].copy()
for c in CONT:
    X[c] = X[c].astype(float).fillna(X[c].median())
    X[c] = (X[c] - X[c].mean()) / (X[c].std() or 1)
for c in CATS:
    X[c] = X[c].astype(str)
X["kyc_verified"] = cohort["kyc_verified"].astype(str).str.lower().isin(["true", "1", "yes"]).astype(int)
formula = "churned ~ " + " + ".join(CONT + BIN + ["kyc_verified"] + [f"C({c})" for c in CATS])
data_m = X.assign(churned=cohort["churned"].astype(int))
logit = smf.logit(formula, data=data_m).fit(disp=False, maxiter=200, method="bfgs")
ors = pd.DataFrame({"odds_ratio": np.exp(logit.params), "ci_low": np.exp(logit.conf_int()[0]),
                    "ci_high": np.exp(logit.conf_int()[1]), "p": logit.pvalues}).drop(index="Intercept")
display(ors.sort_values("odds_ratio", ascending=False).style.format("{:.3f}"))
print(f"Pseudo R²: {logit.prsquared:.3f}")

**Robustness: gradient boosting with permutation importance.** If a factor only matters in the linear
model, it is not trusted as a primary driver.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

Xg = pd.get_dummies(X, columns=CATS, drop_first=False, dtype=float)
yg = cohort["churned"].astype(int).to_numpy()
Xtr, Xte, ytr, yte = train_test_split(Xg, yg, test_size=0.3, random_state=59500, stratify=yg)
gbm = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.06, max_iter=300, random_state=59500).fit(Xtr, ytr)
auc = roc_auc_score(yte, gbm.predict_proba(Xte)[:, 1])
pi = permutation_importance(gbm, Xte, yte, scoring="roc_auc", n_repeats=8, random_state=59500)
imp = pd.Series(pi.importances_mean, index=Xg.columns)
imp_grouped = imp.groupby(lambda c: next((k for k in CATS if c.startswith(k + "_")), c)).sum().sort_values(ascending=False)
try:
    logit_tr = smf.logit(formula, data=data_m.loc[Xtr.index]).fit(disp=False, maxiter=200)
    auc_logit = roc_auc_score(yte, logit_tr.predict(data_m.loc[Xte.index]))
except Exception as e:          # a rare category absent from the training split
    auc_logit = np.nan
print(f"Held-out ROC AUC — gradient boosting: {auc:.3f}, logistic regression: {auc_logit:.3f}")
display(imp_grouped.head(15).to_frame("drop in AUC when shuffled"))
record("gbm_auc", auc)

**Attributable churn.** For each factor that VoltRelay can influence, every rider's churn probability is
re-predicted with that factor moved to its good level (no failure; the best-decile wait, range or pack
health), and the drop in average churn is recorded.

In [ ]:
def attributable(col, good):
    d = X.copy()
    d[col] = good
    return float(logit.predict(X).mean() - logit.predict(d).mean())


def z(col, raw_value):
    raw = cohort[col].astype(float)
    return (raw_value - raw.fillna(raw.median()).mean()) / (raw.fillna(raw.median()).std() or 1)


levers = {
    "Stockout in first 14 days": ("any_stockout_14d", 0),
    "Abandoned in queue in first 14 days": ("any_abandon_14d", 0),
    "Very first attempt failed": ("first_attempt_failed", 0),
    "Share of early attempts that failed": ("service_failure_share_14d", z("service_failure_share_14d", 0)),
    "Long waits (to best decile)": ("median_wait_min_14d", z("median_wait_min_14d", cohort["median_wait_min_14d"].quantile(0.1))),
    "Low range per pack (to best decile)": ("km_per_pack_14d", z("km_per_pack_14d", cohort["km_per_pack_14d"].quantile(0.9))),
    "Worn packs received (to best decile)": ("soh_received_14d", z("soh_received_14d", cohort["soh_received_14d"].quantile(0.9))),
    "Outlier-lot packs received": ("bad_lot_share_14d", z("bad_lot_share_14d", 0)),
    "Gen1 cabinets used": ("gen1_share_14d", z("gen1_share_14d", 0)),
    "Raised a ticket in first 14 days": ("ticket_14d", 0),
    "Price paid (to cheapest decile)": ("paid_per_swap_14d", z("paid_per_swap_14d", cohort["paid_per_swap_14d"].quantile(0.1))),
    "Competitor promotions": ("competitor_promo_share_14d", z("competitor_promo_share_14d", 0)),
    "Competitor site nearby": ("competitor_nearby", 0),
}
base_churn = cohort["churned"].mean()
attr = pd.DataFrame([{"factor": k, "variable": v[0], "churn points removed": attributable(*v),
                      "prevalence": (cohort[v[0]] > 0).mean() if v[0] in BIN else np.nan} for k, v in levers.items()])
attr["share of all churn"] = attr["churn points removed"] / base_churn
attr = attr.merge(ors[["odds_ratio", "ci_low", "ci_high", "p"]], left_on="variable", right_index=True, how="left")
attr["gbm importance rank"] = attr["variable"].map(imp_grouped.rank(ascending=False))


def tier(r):
    sig = (r["p"] < 0.01) and (r["ci_low"] > 1 or r["ci_high"] < 1)
    if sig and r["churn points removed"] >= 0.02 and r["gbm importance rank"] <= 8:
        return "Primary"
    if sig and r["churn points removed"] > 0.003:
        return "Secondary"
    return "Not supported"


attr["verdict"] = attr.apply(tier, axis=1)
attr = attr.sort_values("churn points removed", ascending=False)
display(attr.style.format({"churn points removed": "{:+.1%}", "prevalence": "{:.0%}", "share of all churn": "{:.0%}",
                           "odds_ratio": "{:.2f}", "ci_low": "{:.2f}", "ci_high": "{:.2f}", "p": "{:.3g}",
                           "gbm importance rank": "{:.0f}"}))
for _, r in attr.iterrows():
    record(f"attr::{r['factor']}", r["churn points removed"], r["verdict"])

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 0.45 * len(attr) + 1.2))
av = attr.sort_values("churn points removed")
tier_color = {"Primary": BLUE, "Secondary": "#86b6ef", "Not supported": DEEMPH}
ax.barh(range(len(av)), av["churn points removed"] * 100, height=0.55, color=[tier_color[t] for t in av["verdict"]])
for i, (_, r) in enumerate(av.iterrows()):
    ax.annotate(f"{r['churn points removed']*100:+.1f} pts · {r['verdict']}", (max(r["churn points removed"] * 100, 0), i),
                xytext=(4, 0), textcoords="offset points", va="center", fontsize=8.5)
ax.set_yticks(range(len(av)), av["factor"], fontsize=9)
ax.axvline(0, color=INK2, lw=1)
ax.grid(axis="x")
ax.grid(axis="y", visible=False)
ax.set_xlabel("Percentage points of new-rider churn attributable")
titled(ax, "What drives new-rider churn: primary vs secondary factors",
       f"Churn removed if the factor were at its good level · base churn {base_churn:.0%}")
save(fig, "q6_attributable_churn")

**Counterfactual trend.** If every signup month had experienced the early-failure exposure of the
first quarter, what would retention have been? This isolates how much of the retention decline the
service failures explain.

In [ ]:
q0 = cohort["signup_q"].min()
SVC = ["any_stockout_14d", "any_abandon_14d", "first_attempt_failed", "service_failure_share_14d", "median_wait_min_14d"]
ref_rows = X.loc[cohort["signup_q"] == q0, SVC].to_numpy()
cf = X.copy()
cf[SVC] = ref_rows[np.random.default_rng(59500).integers(0, len(ref_rows), size=len(cf))]
comp = pd.DataFrame({"actual": 1 - cohort["churned"].astype(float),
                     "model": 1 - logit.predict(X), "q1_service_levels": 1 - logit.predict(cf),
                     "q": cohort["signup_q"]}).groupby("q").mean()
display(comp.style.format("{:.1%}"))
record("retention_last_q_actual", comp["actual"].iloc[-1])
record("retention_last_q_if_q1_service", comp["q1_service_levels"].iloc[-1])

## 10 · Decision layer — one currency for every problem

Leadership is choosing where the next operating budget goes. Findings in different units (failure
rates, SoH points, odds ratios) cannot be ranked against each other, so this section converts them into
one currency: **rupees per year**. It answers three questions:

1. **What does one failed swap cost?** Lost revenue when the rider does not come back within the hour,
   plus the future margin lost when an early failure makes a new rider quit.
2. **How many failures does each root cause create?** Counterfactual comparison against the better
   equipment in the same city, hour and conditions.
3. **What is each fix worth, and what should VoltRelay be willing to pay for it?** Expressed as a
   *break-even* value, because the data contains no vendor quotes — inventing capex numbers would be
   guessing.

### 10.1 The cost of a failed swap
**Direct loss.** After a service failure, did the same rider complete a swap anywhere within 60 minutes?
If yes, the revenue was recovered (late, at the rider's cost). If not, that swap's margin was lost.

In [ ]:
seq = sw[["rider_id", "event_ts", "completed", "service_failure", "variable_margin_inr", "amount_charged_inr"]].sort_values(
    ["rider_id", "event_ts"]).reset_index(drop=True)
nxt_done = seq["event_ts"].where(seq["completed"])
nxt_done = nxt_done.groupby(seq["rider_id"], observed=True).bfill()
# bfill finds the next completed swap at or after this row; for a failure row that is strictly later
wait_to_recover = (nxt_done - seq["event_ts"]).dt.total_seconds() / 60
fails = seq[seq["service_failure"]].assign(recover_min=wait_to_recover[seq["service_failure"]])
recovered_1h = (fails["recover_min"] <= 60).mean()
margin_per_swap = sw.loc[sw["completed"], "variable_margin_inr"].mean()
revenue_per_swap = sw.loc[sw["completed"], "amount_charged_inr"].mean()
direct_cost = (1 - recovered_1h) * margin_per_swap
print(f"Failures recovered within 60 min: {recovered_1h:.1%}; median time to recover "
      f"{fails['recover_min'].median():.0f} min. Direct margin lost per failure: ₹{direct_cost:.2f}")
record("failures_recovered_1h", recovered_1h)
record("direct_cost_per_failure_inr", direct_cost)

**Churn loss.** From the retention model: how much does one extra early failure raise a new rider's
probability of leaving, and what is a retained rider worth over the following year? The yearly value is
the observed contribution of retained riders in their second to fourth months, annualised — a
conservative choice because it ignores any growth in usage.

In [ ]:
d_up = X.copy()
d_up["service_failure_share_14d"] = X["service_failure_share_14d"] + (
    1 / cohort["attempts_14d"].clip(lower=1)) / (cohort["service_failure_share_14d"].std() or 1)
d_up["any_stockout_14d"] = 1
churn_lift = float((logit.predict(d_up) - logit.predict(X))[cohort["any_stockout_14d"] == 0].mean())
ret_ids = cohort.index[cohort["retained"]]
later = sw[sw["rider_id"].isin(ret_ids) & (sw["days_in"] >= 30) & (sw["days_in"] < 120) & sw["completed"]]
value_3m = later.groupby("rider_id", observed=True)["contribution_inr"].sum().reindex(ret_ids).fillna(0).mean()
rider_year_value = max(value_3m, 0) * 4
churn_cost = churn_lift * rider_year_value
record("early_failure_churn_lift", churn_lift, "extra churn probability from a first stockout")
record("retained_rider_year_value_inr", rider_year_value)
record("churn_cost_per_early_failure_inr", churn_cost)
print(f"A first stockout raises churn probability by {churn_lift:.1%}. A retained rider is worth "
      f"{inr(rider_year_value)} a year in contribution, so the churn cost of that failure is {inr(churn_cost)}.")

In [ ]:
early_fail_share = len(early[early["service_failure"]]) / max(1, sw["service_failure"].sum())
blended_cost = direct_cost + early_fail_share * churn_cost
record("blended_cost_per_failure_inr", blended_cost)
cost_table = pd.DataFrame({
    "component": ["Margin lost when the rider does not recover within 1 hour",
                  "Future margin lost when a new rider quits after an early failure (per early failure)",
                  "Blended cost of an average service failure"],
    "₹ per failure": [direct_cost, churn_cost, blended_cost]})
display(cost_table.style.format({"₹ per failure": "₹{:,.2f}"}))

### 10.2 Failures by root cause
**Equipment × heat.** For every Gen1 attempt, the model from Core question 3 predicts the failure rate
the same attempt would have had on a Gen2 cabinet (same city, hour, vehicle, location type and heat).
The difference, summed, is the number of failures attributable to Gen1 hardware.

In [ ]:
g1 = sw[sw["charger_generation"].astype(str).eq("Gen1") & sw["temp_band"].notna()]
g1_cells = (g1.groupby(["hot", "city", "hour_bucket", "vehicle_class", "location_type"], observed=True)
            .agg(n=("service_failure", "size"), k=("service_failure", "sum")).reset_index())
for c in ["city", "hour_bucket", "vehicle_class", "location_type"]:
    g1_cells[c] = g1_cells[c].astype(str)
g1_cells["hot"] = g1_cells["hot"].astype(int)
as_gen1 = fit.predict(g1_cells.assign(charger_generation="Gen1"))
as_gen2 = fit.predict(g1_cells.assign(charger_generation="Gen2"))
excess = ((as_gen1 - as_gen2) * g1_cells["n"])
excess_hot = excess[g1_cells["hot"] == 1].sum()
excess_all = excess.sum()
n_gen1 = int(st_open["charger_generation"].eq("Gen1").sum())
years = (sw["event_ts"].max() - sw["event_ts"].min()).days / 365.25
record("gen1_excess_failures", excess_all, "failures vs Gen2 counterfactual, whole period")
record("gen1_excess_failures_hot", excess_hot)
record("gen1_excess_share_of_all_failures", excess_all / sw["service_failure"].sum())
gen1_value_per_cabinet_year = excess_all / years / max(1, n_gen1) * blended_cost
record("gen1_breakeven_inr_per_cabinet_year", gen1_value_per_cabinet_year)
print(f"Gen1 cabinets caused ~{excess_all:,.0f} excess failures ({excess_all / sw['service_failure'].sum():.0%} of all "
      f"service failures); {excess_hot / max(excess_all, 1):.0%} of them in ≥40°C hours. Across {n_gen1} Gen1 cabinets that "
      f"is worth {inr(gen1_value_per_cabinet_year)} per cabinet per year — the most VoltRelay should pay per year to fix one.")

**Battery lots.** The excess wear cost of the outlier lots compared with every other pack of the same
type, and what replacing them early would save.

In [ ]:
bat["pack_type"] = bat["pack_type"].astype(str)
normal_wear = bat[~bat["manufacturing_lot"].isin(BAD_LOTS)].groupby("pack_type")["wear_inr_per_swap"].median()
bb = bat[bat["manufacturing_lot"].isin(BAD_LOTS)].copy()
bb["excess_per_swap"] = bb["wear_inr_per_swap"] - bb["pack_type"].map(normal_wear)
bad_swaps = iss[iss["bad"]].groupby("battery_out_str").size()
bb["swaps"] = bb["battery_id"].map(bad_swaps).fillna(0)
excess_wear_total = float((bb["excess_per_swap"].clip(lower=0) * bb["swaps"]).sum())
recent_rate = iss[iss["bad"] & (iss["event_ts"] >= iss["event_ts"].max() - pd.DateOffset(months=3))].shape[0] * 4
excess_wear_year = float(bb["excess_per_swap"].clip(lower=0).median() * recent_rate)
record("bad_lot_excess_wear_total_inr", excess_wear_total)
record("bad_lot_excess_wear_run_rate_year_inr", excess_wear_year)
record("bad_lot_breakeven_inr_per_pack_year", excess_wear_year / max(1, len(bb)))
print(f"Outlier-lot packs cost {inr(excess_wear_total)} in excess wear over the period, and at the current run-rate "
      f"{inr(excess_wear_year)} a year ({inr(excess_wear_year / max(1, len(bb)))} per pack per year) — before counting "
      f"their effect on range and retention.")

**Partner terms.** What the largest partner's current terms cost relative to the network's other
partners, per year, at its current volume.

In [ ]:
others_cps = part.drop(index=["Independent", largest])
peer_cps = np.average(others_cps["contribution_per_swap"], weights=others_cps["swaps"])
lp = pp[pp["partner_id"] == largest]
lp_year_swaps = lp[lp["event_ts"] >= lp["event_ts"].max() - pd.DateOffset(months=3)].shape[0] * 4
gap_year = (peer_cps - part.loc[largest, "contribution_per_swap"]) * lp_year_swaps
list_per_swap = lp["list_price_inr"].mean()
record("largest_partner_gap_vs_peers_inr_year", gap_year)
record("largest_partner_discount_breakeven_pct",
       float(part.loc[largest, "effective_discount"] - (peer_cps - part.loc[largest, "contribution_per_swap"]) / list_per_swap))
print(f"{part.loc[largest, 'partner_name']} earns {inr(part.loc[largest, 'contribution_per_swap'])} per swap vs "
      f"{inr(peer_cps)} for other partners — {inr(gap_year)} a year at current volume. A discount of about "
      f"{RESULTS['largest_partner_discount_breakeven_pct']['value']:.0%} would bring it level with the other partners.")

### 10.3 The four proposals, scored on one scale
Each proposal is tested against the evidence above: does it address a cause the data points to, how much
of the measured problem could it touch, and what does the evidence say about the alternative?

In [ ]:
capacity_fail = sw.loc[~sw["charger_generation"].astype(str).eq("Gen1"), "service_failure"].sum()
scorecard = pd.DataFrame([
    {"proposal": "More stations",
     "what the evidence says": f"Failures are concentrated (worst 10% of stations = {RESULTS['top10pct_stations_failure_share']['value']:.0%} "
                               f"of failures) and {RESULTS['gen1_excess_share_of_all_failures']['value']:.0%} of all failures are "
                               f"excess Gen1 failures in the same conditions; see the expansion-wave table for where new sites went.",
     "addresses a root cause?": "Partly — only where demand exceeds capacity",
     "better alternative": f"Fix or cool the {n_gen1} Gen1 cabinets first; worth up to {inr(gen1_value_per_cabinet_year)} per cabinet per year"},
    {"proposal": "More batteries",
     "what the evidence says": f"Outlier lots {', '.join(sorted(BAD_LOTS)) or 'n/a'} degrade "
                               f"{RESULTS['bad_lot_degradation_multiple']['value']:.1f}× faster; excess wear {inr(excess_wear_year)}/year.",
     "addresses a root cause?": "Only if it replaces the outlier lots",
     "better alternative": "Replace (and claim warranty on) outlier-lot packs rather than adding more of the same"},
    {"proposal": "Network-wide pricing rollout",
     "what the evidence says": f"Pilot DiD on peak share: {RESULTS['did::peak_share']['value']:+.2%}; on peak failure rate: "
                               f"{RESULTS['did::peak_fail']['value']:+.2%} (see 8.2 for confidence intervals).",
     "addresses a root cause?": "Only where peak congestion is real",
     "better alternative": "Targeted peak pricing at congested stations and hours, not network-wide"},
    {"proposal": f"Long-term exclusive with {part.loc[largest, 'partner_name']}",
     "what the evidence says": f"Largest partner by volume ({RESULTS['largest_partner_swap_share']['value']:.0%} of swaps) earns "
                               f"{inr(part.loc[largest, 'contribution_per_swap'])} per swap vs {inr(peer_cps)} for other partners.",
     "addresses a root cause?": "No — it locks in the weakest terms",
     "better alternative": "Renegotiate first: discount floor near break-even, no exclusivity until margin is level"},
]).set_index("proposal")
display(scorecard)
scorecard.to_csv(OUT_DIR / "budget_scorecard.csv")

### 10.4 Support tickets: does the free text add anything?
The re-labelling in 3.8 found battery and range complaints hiding under generic categories. Two checks:
do those complaints track the outlier battery lots, and how does the complaint mix move over time?

In [ ]:
tk["battery_lot"] = tk["battery_id"].map(bat.set_index("battery_id")["manufacturing_lot"])
with_pack = tk[tk["battery_lot"].notna()]
lot_view = with_pack.assign(bad_lot=with_pack["battery_lot"].isin(BAD_LOTS)).groupby("bad_lot").agg(
    tickets=("ticket_id", "size"), battery_or_range_share=("mentions_battery", "mean"))
display(lot_view.rename(index={True: "outlier-lot pack", False: "other pack"}))
theme = tk.groupby([tk["created_ts"].dt.to_period("Q"), "category_adj"]).size().unstack(fill_value=0)
theme = theme.div(theme.sum(axis=1), axis=0)
theme.index = theme.index.astype(str)
display(theme.style.format("{:.0%}"))

### 10.5 Anomalies worth flagging separately
Station-months whose failure rate is extreme relative to the network that month (robust z-score above
4), plus the irregularities already found during cleaning.

In [ ]:
sm_ = sw.groupby(["station_id", "month"], observed=True).agg(attempts=("service_failure", "size"),
                                                             rate=("service_failure", "mean")).reset_index()
sm_ = sm_[sm_["attempts"] >= 200]
med_m = sm_.groupby("month")["rate"].transform("median")
mad_m = sm_.groupby("month")["rate"].transform(lambda s: (s - s.median()).abs().median() * 1.4826).replace(0, np.nan)
sm_["z"] = (sm_["rate"] - med_m) / mad_m
extreme = sm_[sm_["z"] > 4].merge(st[["station_id", "city", "charger_generation", "location_type", "connectivity_tier"]],
                                  on="station_id", how="left").sort_values("z", ascending=False)
print(f"{len(extreme)} extreme station-months across {extreme['station_id'].nunique()} stations")
display(extreme.head(15))
anomalies = pd.DataFrame([
    ("Test stations booking transactions", RESULTS["test_station_events"]["value"], "Exclude from reporting; audit why revenue is booked"),
    ("Firmware v3.2.0 clock bug", RESULTS["clock_bug_events"]["value"], "Timestamps corrected; fix firmware time zone"),
    ("Offline-sync duplicate records", RESULTS["duplicates_removed"]["value"], "De-duplicated; add idempotent event IDs at sync"),
    ("Packs issued after recorded retirement", RESULTS["retired_packs_issued"]["value"], "Safety check: block retired IDs at the cabinet"),
    ("Extreme station-months (z > 4)", len(extreme), "Review with operations; most are listed above"),
], columns=["anomaly", "rows / cases", "action"])
display(anomalies)
anomalies.to_csv(OUT_DIR / "anomalies.csv", index=False)

## 11 · Key findings

The table below is generated from the numbers computed above, so it cannot drift from the analysis.
The written report and the video use exactly these values.

In [ ]:
def v(k):
    return RESULTS[k]["value"] if k in RESULTS else np.nan


findings = pd.DataFrame([
    ("Performance", f"Completed swaps grew {v('growth_completed_x'):.1f}× and revenue {v('growth_revenue_x'):.1f}×, "
                    f"but service failures grew {v('growth_failures_x'):.1f}× (rate {pct(v('failure_rate_first3'))} → "
                    f"{pct(v('failure_rate_last3'))}); contribution per swap went from ₹{v('contribution_per_swap_first3'):.1f} "
                    f"to ₹{v('contribution_per_swap_last3'):.1f}."),
    ("Failures", f"The worst 10% of stations produce {pct(v('top10pct_stations_failure_share'), 0)} of service failures "
                 f"on {pct(v('top10pct_stations_attempt_share'), 0)} of attempts."),
    ("Equipment × heat", f"Model-adjusted failure rate: Gen1 {pct(v('adj_fail::Gen1::normal'))} normally vs "
                         f"{pct(v('adj_fail::Gen1::hot'))} in ≥40°C hours; Gen2 {pct(v('adj_fail::Gen2::normal'))} vs "
                         f"{pct(v('adj_fail::Gen2::hot'))}. Gen1 accounts for {v('gen1_excess_failures'):,.0f} excess failures "
                         f"({pct(v('gen1_excess_share_of_all_failures'), 0)} of all)."),
    ("Batteries", f"Lots {v('bad_lots')} degrade {v('bad_lot_degradation_multiple'):.1f}× faster than all other lots; "
                  f"excess wear {inr(v('bad_lot_excess_wear_total_inr'))} over the period."),
    ("Pricing", f"Pilot difference-in-differences: peak-hour share {v('did::peak_share'):+.2%}, peak failure rate "
                f"{v('did::peak_fail'):+.2%}, revenue per swap ₹{v('did::rev_per_swap'):+.2f}."),
    ("Partners", f"Largest partner {v('largest_partner')} ({pct(v('largest_partner_swap_share'), 0)} of swaps) earns "
                 f"₹{v('largest_partner_contribution_per_swap'):.1f} per swap vs ₹{v('independent_contribution_per_swap'):.1f} "
                 f"from independent riders."),
    ("Retention", f"Month-2 retention {pct(v('retention_no_stockout'))} without an early stockout vs "
                  f"{pct(v('retention_with_stockout'))} with one; latest-quarter retention {pct(v('retention_last_q_actual'))} "
                  f"would be {pct(v('retention_last_q_if_q1_service'))} at first-quarter service levels."),
    ("Cost of a failure", f"A service failure costs ₹{v('direct_cost_per_failure_inr'):.1f} directly; an early failure for a "
                          f"new rider costs {inr(v('churn_cost_per_early_failure_inr'))} in lost future margin."),
], columns=["area", "finding"])
with pd.option_context("display.max_colwidth", 300):
    display(findings)

In [ ]:
out = {k: v_["value"] for k, v_ in RESULTS.items()}
(OUT_DIR / "results.json").write_text(json.dumps(out, indent=1, default=str))
monthly.to_csv(OUT_DIR / "monthly_network.csv")
attr.to_csv(OUT_DIR / "churn_drivers.csv", index=False)
part.to_csv(OUT_DIR / "partner_economics.csv")
lots.to_csv(OUT_DIR / "battery_lots.csv", index=False)
print("RESULTS_BLOCK_START")
print(json.dumps(out, default=str))
print("RESULTS_BLOCK_END")
print(f"\nFigures saved to {FIG_DIR.resolve()} ({len(list(FIG_DIR.glob('*.png')))} files); tables to {OUT_DIR.resolve()}")